In [ ]:

plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (12, 8)


In [ ]:
# ============================================================================
# 第一部分：数据预处理 - 增强版Multi-Subject-Out分割
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.stats import f_oneway, kruskal
import h5py
import time
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (12, 8)


def load_and_prepare_data_multi_subject_out(data_path='/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38.mat',
                                          train_subjects_range=(1, 31),    # 受试者1-30用于训练
                                          val_subjects_range=(31, 38),     # 受试者31-37用于验证  
                                          test_subject=38,                 # 受试者38用于测试
                                          random_state=42):
    """
    Multi-Subject-Out数据分割策略
    
    Args:
        data_path: 数据文件路径
        train_subjects_range: 训练集受试者范围 (start, end) - 左闭右开
        val_subjects_range: 验证集受试者范围 (start, end) - 左闭右开  
        test_subject: 测试集受试者ID
        random_state: 随机种子
        
    Returns:
        dict: 包含所有数据和受试者信息的字典
    """
    
    print("📂 加载数据 - Multi-Subject-Out策略...")
    
    # 加载原始数据
    f = h5py.File(data_path, 'r')
    arrays = {}
    for k, v in f.items():
        arrays[k] = np.array(v)
    f.close()
    
    train_data = arrays['data'].transpose()
    train_region = arrays['region'].transpose()
    prob_idx = arrays['prob_idx'].transpose().flatten()  # 确保是1D数组
    
    print(f"原始数据形状: {train_data.shape}")
    print(f"原始标签形状: {train_region.shape}")
    print(f"prob_idx形状: {prob_idx.shape}")
    print(f"受试者ID范围: {np.min(prob_idx)} - {np.max(prob_idx)}")
    
    del arrays, f
    
    # 按受试者分割数据
    print(f"\n🎯 按Multi-Subject-Out策略分割数据...")
    print(f"  训练集: 受试者 {train_subjects_range[0]}-{train_subjects_range[1]-1}")
    print(f"  验证集: 受试者 {val_subjects_range[0]}-{val_subjects_range[1]-1}")  
    print(f"  测试集: 受试者 {test_subject}")
    
    # 创建受试者掩码
    train_subjects_ids = list(range(train_subjects_range[0], train_subjects_range[1]))
    val_subjects_ids = list(range(val_subjects_range[0], val_subjects_range[1]))
    
    train_mask = np.isin(prob_idx, train_subjects_ids)
    val_mask = np.isin(prob_idx, val_subjects_ids)
    test_mask = prob_idx == test_subject
    
    # 验证分割完整性
    total_samples = len(prob_idx)
    assigned_samples = np.sum(train_mask) + np.sum(val_mask) + np.sum(test_mask)
    
    print(f"\n📊 分割结果验证:")
    print(f"  总样本数: {total_samples:,}")
    print(f"  已分配样本数: {assigned_samples:,}")
    print(f"  未分配样本数: {total_samples - assigned_samples:,}")
    
    if assigned_samples != total_samples:
        missing_subjects = set(np.unique(prob_idx)) - set(train_subjects_ids + val_subjects_ids + [test_subject])
        print(f"  ⚠️ 警告: 存在未分配的受试者: {sorted(missing_subjects)}")
    
    # 提取各数据集
    X_train = train_data[train_mask]
    y_train = train_region[train_mask] 
    subjects_train = prob_idx[train_mask]
    
    X_val = train_data[val_mask]
    y_val = train_region[val_mask]
    subjects_val = prob_idx[val_mask]
    
    X_test = train_data[test_mask]
    y_test = train_region[test_mask]
    subjects_test = prob_idx[test_mask]
    
    print(f"\n📈 最终数据集统计:")
    print(f"  训练集: {X_train.shape[0]:,} 样本, {len(np.unique(subjects_train))} 个受试者")
    print(f"  验证集: {X_val.shape[0]:,} 样本, {len(np.unique(subjects_val))} 个受试者") 
    print(f"  测试集: {X_test.shape[0]:,} 样本, {len(np.unique(subjects_test))} 个受试者")
    
    # 受试者样本分布统计
    print(f"\n👥 受试者样本分布:")
    
    print("  训练集受试者:")
    for subject_id in sorted(np.unique(subjects_train)):
        count = np.sum(subjects_train == subject_id)
        print(f"    受试者{subject_id}: {count:,} 样本")
    
    print("  验证集受试者:")
    for subject_id in sorted(np.unique(subjects_val)):
        count = np.sum(subjects_val == subject_id)
        print(f"    受试者{subject_id}: {count:,} 样本")
    
    test_count = np.sum(subjects_test == test_subject)
    print(f"  测试集受试者{test_subject}: {test_count:,} 样本")
    
    # 清理内存
    del train_data, train_region, prob_idx
    
    # 应用标准化
    print(f"\n📊 应用标准化...")
    scaler = StandardScaler()
    scaler.fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    print("✅ Multi-Subject-Out数据准备完成")
    
    # 返回完整的数据字典
    return {
        # 原始数据
        'X_train': X_train, 'y_train': y_train, 'subjects_train': subjects_train,
        'X_val': X_val, 'y_val': y_val, 'subjects_val': subjects_val,
        'X_test': X_test, 'y_test': y_test, 'subjects_test': subjects_test,
        
        # 标准化数据
        'X_train_scaled': X_train_scaled,
        'X_val_scaled': X_val_scaled, 
        'X_test_scaled': X_test_scaled,
        
        # 元信息
        'scaler': scaler,
        'train_subjects_ids': train_subjects_ids,
        'val_subjects_ids': val_subjects_ids,
        'test_subject_id': test_subject,
        'feature_dim': X_train.shape[1],
        'n_classes': y_train.shape[1] if len(y_train.shape) > 1 else len(np.unique(np.argmax(y_train, axis=1))),
        
        # 分割策略信息
        'split_strategy': 'Multi-Subject-Out',
        'split_config': {
            'train_subjects_range': train_subjects_range,
            'val_subjects_range': val_subjects_range, 
            'test_subject': test_subject
        }
    }

In [ ]:
class SubjectEmbeddingAnalyzer:
    """
    Subject Embedding 可行性分析器
    
    完整分析流程：
    Phase 1: 受试者间差异分析
    Phase 2: 受试者可分离性评估  
    Phase 3: Embedding适配性评估
    Phase 4: 决策建议生成
    """
    
    def __init__(self, save_path='./subject_embedding_analysis/'):
        self.save_path = save_path
        os.makedirs(save_path, exist_ok=True)
        os.makedirs(os.path.join(save_path, 'visualizations'), exist_ok=True)
        
        # 分析结果存储
        self.analysis_results = {}
        self.decision_scores = {}
        
        print("🧠 Subject Embedding 可行性分析器初始化完成")
        print(f"📁 结果保存路径: {save_path}")


    def prepare_data_with_subjects_enhanced(self, data_dict):
        """
        增强版数据准备 - 直接使用Multi-Subject-Out的完整受试者信息
        
        Args:
            data_dict: load_and_prepare_data_multi_subject_out()的返回结果
            
        Returns:
            dict: 包含完整受试者映射的数据字典
        """
        print("\n" + "="*80)
        print("📊 Phase 0: 增强版数据准备（基于Multi-Subject-Out）")
        print("="*80)
        
        # 验证数据字典完整性
        required_keys = ['X_train_scaled', 'y_train', 'subjects_train',
                        'X_val_scaled', 'y_val', 'subjects_val', 
                        'X_test_scaled', 'y_test', 'subjects_test']
        
        missing_keys = [key for key in required_keys if key not in data_dict]
        if missing_keys:
            raise ValueError(f"数据字典缺少必要字段: {missing_keys}")
        
        # 提取数据
        X_train = data_dict['X_train_scaled']
        y_train = data_dict['y_train'] 
        subjects_train = data_dict['subjects_train']
        
        X_val = data_dict['X_val_scaled']
        y_val = data_dict['y_val']
        subjects_val = data_dict['subjects_val']
        
        X_test = data_dict['X_test_scaled'] 
        y_test = data_dict['y_test']
        subjects_test = data_dict['subjects_test']
        
        # 获取所有可用受试者
        all_subjects = np.concatenate([subjects_train, subjects_val, subjects_test])
        available_subjects = np.unique(all_subjects)
        
        print(f"✅ 数据验证通过")
        print(f"  - 训练集: {len(subjects_train):,} 样本, 受试者 {sorted(np.unique(subjects_train))}")
        print(f"  - 验证集: {len(subjects_val):,} 样本, 受试者 {sorted(np.unique(subjects_val))}")
        print(f"  - 测试集: {len(subjects_test):,} 样本, 受试者 {sorted(np.unique(subjects_test))}")
        print(f"  - 总受试者数: {len(available_subjects)}")
        
        # 计算受试者样本统计
        subject_counts = {}
        for subject_id in available_subjects:
            train_count = np.sum(subjects_train == subject_id)
            val_count = np.sum(subjects_val == subject_id) 
            test_count = np.sum(subjects_test == subject_id)
            total_count = train_count + val_count + test_count
            
            subject_counts[subject_id] = {
                'train': train_count,
                'val': val_count, 
                'test': test_count,
                'total': total_count
            }
        
        # 存储完整数据
        self.data = {
            'X_train': X_train, 'y_train': y_train, 'subjects_train': subjects_train,
            'X_val': X_val, 'y_val': y_val, 'subjects_val': subjects_val,
            'X_test': X_test, 'y_test': y_test, 'subjects_test': subjects_test,
            'available_subjects': available_subjects,
            'subject_counts': subject_counts,
            'split_strategy': data_dict.get('split_strategy', 'Multi-Subject-Out'),
            'split_config': data_dict.get('split_config', {}),
            'mapping_method': 'precise_multi_subject_out'
        }
        
        print(f"✅ 增强版数据准备完成")
        print(f"  - 数据映射方法: 精确Multi-Subject-Out")
        
        return self.data



    def phase1_subject_differences_analysis(self):
        """
        Phase 1: 受试者间差异本质分析 - 修正版 (增强数值稳定性)
        """
        print("\n" + "="*80)
        print("📊 Phase 1: 受试者间差异本质分析")
        print("="*80)
        
        # 1.1 统计分布差异诊断
        print("\n🔍 1.1 计算受试者统计特征...")
        
        subjects = self.data['available_subjects']
        n_features = self.data['X_train'].shape[1]
        
        # 为每个受试者计算统计量
        subject_stats = {
            'means': np.zeros((len(subjects), n_features)),
            'stds': np.zeros((len(subjects), n_features)),
            'skews': np.zeros((len(subjects), n_features)),
            'kurts': np.zeros((len(subjects), n_features)),
            'sample_counts': np.zeros(len(subjects))
        }
        
        for i, subject_id in enumerate(subjects):
            # 训练集中该受试者的数据
            subject_mask = self.data['subjects_train'] == subject_id
            if np.sum(subject_mask) > 0:
                subject_data = self.data['X_train'][subject_mask]
                
                subject_stats['means'][i] = np.mean(subject_data, axis=0)
                subject_stats['stds'][i] = np.std(subject_data, axis=0)
                subject_stats['skews'][i] = stats.skew(subject_data, axis=0)
                subject_stats['kurts'][i] = stats.kurtosis(subject_data, axis=0)
                subject_stats['sample_counts'][i] = len(subject_data)
            
            # 处理验证集
            val_mask = self.data['subjects_val'] == subject_id
            if np.sum(val_mask) > 0:
                val_data = self.data['X_val'][val_mask]
                subject_stats['sample_counts'][i] += len(val_data)
        
        self.analysis_results['subject_stats'] = subject_stats
        
        print(f"✅ 受试者统计特征计算完成")
        print(f"  - 平均每受试者样本量: {np.mean(subject_stats['sample_counts']):.0f}")
        print(f"  - 样本量范围: [{np.min(subject_stats['sample_counts']):.0f}, {np.max(subject_stats['sample_counts']):.0f}]")
        
        # 1.2 增强版受试者间相似性分析
        print("\n🔍 1.2 分析受试者间相似性 (增强数值稳定性)...")
        
        subject_means = subject_stats['means']
        
        # 🔧 新增：数值稳定性检查和处理
        print("    🔍 执行数值稳定性检查...")
        
        # 检查常数特征
        feature_variance = np.var(subject_means, axis=0)
        constant_features = np.sum(feature_variance < 1e-12)
        valid_features_mask = feature_variance >= 1e-12
        
        print(f"    - 总特征数: {n_features}")
        print(f"    - 常数特征数: {constant_features}")
        print(f"    - 有效特征数: {np.sum(valid_features_mask)}")
        
        if constant_features > 0:
            print(f"    ⚠️ 发现{constant_features}个常数特征，将在相关性计算中排除")
            subject_means_filtered = subject_means[:, valid_features_mask]
        else:
            subject_means_filtered = subject_means
        
        # 检查NaN和无穷值
        nan_count = np.sum(np.isnan(subject_means_filtered))
        inf_count = np.sum(np.isinf(subject_means_filtered))
        
        if nan_count > 0 or inf_count > 0:
            print(f"    ⚠️ 发现异常值: NaN({nan_count}), Inf({inf_count})")
            # 替换异常值
            subject_means_filtered = np.nan_to_num(subject_means_filtered, 
                                                nan=0.0, posinf=1e10, neginf=-1e10)
            print(f"    ✅ 异常值已处理")
        
        # 🔧 修复：鲁棒的距离计算
        try:
            print("    🔍 计算受试者间距离矩阵...")
            distance_matrix = squareform(pdist(subject_means_filtered, metric='euclidean'))
            distance_computation_success = True
            
            # 距离矩阵统计
            upper_triangle_distances = distance_matrix[np.triu_indices_from(distance_matrix, k=1)]
            print(f"    - 距离矩阵形状: {distance_matrix.shape}")
            print(f"    - 最小距离: {np.min(upper_triangle_distances):.6f}")
            print(f"    - 最大距离: {np.max(upper_triangle_distances):.6f}")
            print(f"    - 平均距离: {np.mean(upper_triangle_distances):.6f}")
            print(f"    - 距离标准差: {np.std(upper_triangle_distances):.6f}")
            
        except Exception as e:
            print(f"    ❌ 距离计算失败: {e}")
            distance_matrix = np.zeros((len(subjects), len(subjects)))
            distance_computation_success = False
        
        # 🔧 修复：鲁棒的相关性计算
        correlation_computation_success = False
        correlation_matrix = np.full((len(subjects), len(subjects)), np.nan)
        
        try:
            print("    🔍 计算受试者间相关性矩阵...")
            
            # 方法1: 标准相关性计算
            if subject_means_filtered.shape[1] > 1 and len(subjects) > 1:
                # 标准化数据以提高数值稳定性
                subject_means_normalized = stats.zscore(subject_means_filtered, axis=1, nan_policy='omit')
                
                # 替换可能的NaN
                subject_means_normalized = np.nan_to_num(subject_means_normalized, nan=0.0)
                
                correlation_matrix = np.corrcoef(subject_means_normalized)
                
                # 验证相关性矩阵的有效性
                if np.all(np.isfinite(correlation_matrix)):
                    correlation_computation_success = True
                    
                    # 相关性矩阵统计
                    upper_triangle_corr = correlation_matrix[np.triu_indices_from(correlation_matrix, k=1)]
                    valid_correlations = upper_triangle_corr[np.isfinite(upper_triangle_corr)]
                    
                    print(f"    - 相关性矩阵形状: {correlation_matrix.shape}")
                    print(f"    - 有效相关性数量: {len(valid_correlations)} / {len(upper_triangle_corr)}")
                    
                    if len(valid_correlations) > 0:
                        print(f"    - 最小相关性: {np.min(valid_correlations):.6f}")
                        print(f"    - 最大相关性: {np.max(valid_correlations):.6f}")
                        print(f"    - 平均相关性: {np.mean(valid_correlations):.6f}")
                        print(f"    - 相关性标准差: {np.std(valid_correlations):.6f}")
                    else:
                        print(f"    ⚠️ 没有有效的相关性值")
                else:
                    raise ValueError("相关性矩阵包含无效值")
                    
        except Exception as e:
            print(f"    ⚠️ 标准相关性计算失败: {e}")
            
            # 方法2: 备用相关性计算
            try:
                print("    🔍 尝试备用相关性计算方法...")
                correlation_matrix = np.zeros((len(subjects), len(subjects)))
                
                for i in range(len(subjects)):
                    for j in range(i, len(subjects)):
                        if i == j:
                            correlation_matrix[i, j] = 1.0
                        else:
                            # 使用Spearman相关性作为备用
                            try:
                                corr, _ = stats.spearmanr(subject_means_filtered[i], subject_means_filtered[j])
                                if np.isfinite(corr):
                                    correlation_matrix[i, j] = correlation_matrix[j, i] = corr
                                else:
                                    correlation_matrix[i, j] = correlation_matrix[j, i] = 0.0
                            except:
                                correlation_matrix[i, j] = correlation_matrix[j, i] = 0.0
                
                correlation_computation_success = True
                print(f"    ✅ 备用相关性计算成功")
                
                # 统计备用方法的结果
                upper_triangle_corr = correlation_matrix[np.triu_indices_from(correlation_matrix, k=1)]
                valid_correlations = upper_triangle_corr[np.isfinite(upper_triangle_corr)]
                
                if len(valid_correlations) > 0:
                    print(f"    - 平均相关性 (Spearman): {np.mean(valid_correlations):.6f}")
                    print(f"    - 相关性范围: [{np.min(valid_correlations):.6f}, {np.max(valid_correlations):.6f}]")
                
            except Exception as e2:
                print(f"    ❌ 备用相关性计算也失败: {e2}")
                correlation_matrix = np.eye(len(subjects))  # 使用单位矩阵作为最后的备用
        
        # 🔧 修复：鲁棒的层次聚类
        try:
            print("    🔍 计算层次聚类...")
            
            if distance_computation_success and len(subjects) > 1:
                # 使用距离矩阵进行聚类
                condensed_distances = squareform(distance_matrix)
                linkage_matrix = linkage(condensed_distances, method='ward')
                print(f"    ✅ 层次聚类计算成功")
            else:
                # 使用原始数据进行聚类
                linkage_matrix = linkage(subject_means_filtered, method='ward')
                print(f"    ✅ 层次聚类计算成功 (使用原始数据)")
                
        except Exception as e:
            print(f"    ⚠️ 层次聚类计算失败: {e}")
            # 创建一个简单的层次结构作为备用
            linkage_matrix = np.zeros((len(subjects)-1, 4))
            for i in range(len(subjects)-1):
                linkage_matrix[i] = [i, i+1, 1.0, 2]
        
        # 保存相似性分析结果
        self.analysis_results['subject_similarity'] = {
            'distance_matrix': distance_matrix,
            'correlation_matrix': correlation_matrix,
            'linkage_matrix': linkage_matrix,
            'distance_computation_success': distance_computation_success,
            'correlation_computation_success': correlation_computation_success,
            'constant_features_count': constant_features,
            'valid_features_count': np.sum(valid_features_mask),
            'numerical_issues': {
                'nan_count': nan_count,
                'inf_count': inf_count,
                'constant_features': constant_features
            }
        }
        
        # 输出总结
        if distance_computation_success:
            avg_distance = np.mean(distance_matrix[np.triu_indices_from(distance_matrix, k=1)])
            print(f"✅ 受试者相似性分析完成")
            print(f"  - 平均受试者间距离: {avg_distance:.4f}")
        else:
            print(f"⚠️ 受试者距离计算遇到问题")
        
        if correlation_computation_success:
            upper_triangle_corr = correlation_matrix[np.triu_indices_from(correlation_matrix, k=1)]
            valid_correlations = upper_triangle_corr[np.isfinite(upper_triangle_corr)]
            if len(valid_correlations) > 0:
                avg_correlation = np.mean(valid_correlations)
                print(f"  - 平均受试者间相关性: {avg_correlation:.4f}")
            else:
                print(f"  - 平均受试者间相关性: 无有效值")
        else:
            print(f"  - 平均受试者间相关性: 计算失败")
        
        # 1.3 特征变异模式分析 (保持原有逻辑)
        print("\n🔍 1.3 分析特征变异模式...")
        
        # 计算每个特征在不同受试者间的方差
        feature_f_stats = np.var(subject_means, axis=0)  # 受试者间方差
        feature_f_stats_norm = feature_f_stats / (np.mean(feature_f_stats) + 1e-8)  # 防除零
        
        # 识别高变异和低变异特征
        high_variation_threshold = np.percentile(feature_f_stats_norm, 90)
        low_variation_threshold = np.percentile(feature_f_stats_norm, 10)
        
        high_variation_features = np.where(feature_f_stats_norm > high_variation_threshold)[0]
        low_variation_features = np.where(feature_f_stats_norm < low_variation_threshold)[0]
        
        # PCA分析受试者差异模式
        pca = PCA()
        subject_pca_result = pca.fit_transform(subject_means)
        
        self.analysis_results['feature_variation'] = {
            'f_stats': feature_f_stats_norm,
            'high_variation_features': high_variation_features,
            'low_variation_features': low_variation_features,
            'pca_result': subject_pca_result,
            'pca_explained_variance': pca.explained_variance_ratio_,
            'pca_cumulative_variance': np.cumsum(pca.explained_variance_ratio_)
        }
        
        print(f"✅ 特征变异分析完成")
        print(f"  - 高变异特征数量: {len(high_variation_features)} ({len(high_variation_features)/n_features*100:.1f}%)")
        print(f"  - 低变异特征数量: {len(low_variation_features)} ({len(low_variation_features)/n_features*100:.1f}%)")
        print(f"  - 前3个主成分解释方差: {np.sum(pca.explained_variance_ratio_[:3])*100:.1f}%")
        print(f"  - 前10个主成分解释方差: {np.sum(pca.explained_variance_ratio_[:10])*100:.1f}%")
        
        # 决策得分计算
        variance_explained_3pc = np.sum(pca.explained_variance_ratio_[:3])
        
        # 🔧 修复：处理相关性可能为nan的情况
        if correlation_computation_success:
            upper_triangle_corr = correlation_matrix[np.triu_indices_from(correlation_matrix, k=1)]
            valid_correlations = upper_triangle_corr[np.isfinite(upper_triangle_corr)]
            if len(valid_correlations) > 0:
                mean_correlation = np.mean(valid_correlations)
            else:
                mean_correlation = 0.0  # 默认值
        else:
            mean_correlation = 0.0  # 默认值
        
        # Phase 1 决策得分
        self.decision_scores['phase1'] = {
            'difference_significance': 1.0 if variance_explained_3pc > 0.6 else 0.5,
            'pattern_linearity': variance_explained_3pc,
            'subject_similarity': abs(mean_correlation),
            'feature_heterogeneity': len(high_variation_features) / n_features
        }
        
        print(f"\n📈 Phase 1 决策指标:")
        print(f"  - 差异显著性得分: {self.decision_scores['phase1']['difference_significance']:.3f}")
        print(f"  - 模式线性度: {self.decision_scores['phase1']['pattern_linearity']:.3f}")
        print(f"  - 受试者相似性: {self.decision_scores['phase1']['subject_similarity']:.3f}")
        print(f"  - 特征异质性: {self.decision_scores['phase1']['feature_heterogeneity']:.3f}")
            
    def phase2_subject_separability_analysis(self):
        """
        Phase 2: 受试者可分离性评估 - 修复版
        """
        print("\n" + "="*80)
        print("📊 Phase 2: 受试者可分离性评估")
        print("="*80)
        
        # 2.1 增强版受试者识别难度测试
        print("\n🔍 2.1 受试者识别难度测试 (增强版)...")
        
        # 准备受试者识别数据
        X_subject_id = self.data['X_train']
        y_subject_id = self.data['subjects_train']
        
        # 确保受试者ID是整数类型
        y_subject_id = y_subject_id.astype(int)
        
        # 数据清理和映射
        print("    🔍 数据清理和预处理...")
        
        # 获取受试者统计信息
        unique_subjects, subject_counts = np.unique(y_subject_id, return_counts=True)
        print(f"    - 原始受试者数量: {len(unique_subjects)}")
        print(f"    - 受试者样本量范围: [{np.min(subject_counts)}, {np.max(subject_counts)}]")
        print(f"    - 平均样本量: {np.mean(subject_counts):.0f}")
        
        # 🔧 修复：使用更合理的样本量阈值
        min_samples_threshold = max(1000, np.mean(subject_counts) * 0.1)  # 动态阈值
        valid_subjects = unique_subjects[subject_counts >= min_samples_threshold]
        
        print(f"    - 样本量阈值: {min_samples_threshold:.0f}")
        print(f"    - 有效受试者数量: {len(valid_subjects)}")
        
        if len(valid_subjects) < 3:
            print("    ⚠️ 有效受试者数量不足，降低阈值重试...")
            min_samples_threshold = max(500, np.min(subject_counts[subject_counts > 0]))
            valid_subjects = unique_subjects[subject_counts >= min_samples_threshold]
            print(f"    - 调整后阈值: {min_samples_threshold:.0f}")
            print(f"    - 调整后有效受试者数量: {len(valid_subjects)}")
        
        if len(valid_subjects) >= 3:
            # 筛选数据
            valid_mask = np.isin(y_subject_id, valid_subjects)
            X_subset = X_subject_id[valid_mask]
            y_subset = y_subject_id[valid_mask]
            
            # 重新编码受试者ID到连续范围
            subject_mapping = {old_id: new_id for new_id, old_id in enumerate(valid_subjects)}
            y_subset_remapped = np.array([subject_mapping[old_id] for old_id in y_subset])
            
            print(f"    - 筛选后样本数: {len(X_subset):,}")
            print(f"    - 受试者ID重新映射: {len(subject_mapping)} 个受试者")
            
            # 🔧 修复：智能采样策略
            max_samples_per_subject = 5000
            max_total_samples = 200000  # 总样本数限制
            
            # 计算每个受试者应该采样的数量
            unique_remapped, remapped_counts = np.unique(y_subset_remapped, return_counts=True)
            
            if len(X_subset) > max_total_samples:
                print(f"    🔍 数据量过大，进行智能分层采样...")
                
                # 计算采样比例
                total_ratio = max_total_samples / len(X_subset)
                samples_per_subject = {}
                
                for subject_id, count in zip(unique_remapped, remapped_counts):
                    target_samples = min(max_samples_per_subject, int(count * total_ratio))
                    target_samples = max(100, target_samples)  # 确保每个受试者至少100个样本
                    samples_per_subject[subject_id] = target_samples
                
                # 分层采样
                sampled_indices = []
                for subject_id in unique_remapped:
                    subject_mask = y_subset_remapped == subject_id
                    subject_indices = np.where(subject_mask)[0]
                    
                    target_samples = samples_per_subject[subject_id]
                    if len(subject_indices) > target_samples:
                        # 随机采样
                        sampled_subject_indices = np.random.choice(
                            subject_indices, target_samples, replace=False
                        )
                        sampled_indices.extend(sampled_subject_indices)
                    else:
                        sampled_indices.extend(subject_indices)
                
                X_subset = X_subset[sampled_indices]
                y_subset_remapped = y_subset_remapped[sampled_indices]
                
                print(f"    - 采样后样本数: {len(X_subset):,}")
                
                # 验证采样后的分布
                final_unique, final_counts = np.unique(y_subset_remapped, return_counts=True)
                print(f"    - 采样后受试者数量: {len(final_unique)}")
                print(f"    - 采样后样本量范围: [{np.min(final_counts)}, {np.max(final_counts)}]")
                print(f"    - 采样后平均样本量: {np.mean(final_counts):.0f}")
            
            # 🔧 修复：受试者感知的交叉验证
            try:
                print("    🔍 执行受试者感知的交叉验证...")
                
                # 计算合适的CV折数
                n_unique_subjects = len(np.unique(y_subset_remapped))
                cv_folds = min(5, max(3, n_unique_subjects // 2))  # 确保每折至少有一些受试者
                
                print(f"    - 受试者数量: {n_unique_subjects}")
                print(f"    - CV折数: {cv_folds}")
                
                # 🔧 新增：自定义受试者级别的交叉验证
                def subject_aware_cv_split(y_subjects, cv_folds):
                    """受试者感知的交叉验证分割"""
                    unique_subjects = np.unique(y_subjects)
                    np.random.shuffle(unique_subjects)  # 随机打乱受试者顺序
                    
                    fold_subjects = np.array_split(unique_subjects, cv_folds)
                    
                    for fold_idx in range(cv_folds):
                        test_subjects = fold_subjects[fold_idx]
                        train_subjects = np.concatenate([fold_subjects[i] for i in range(cv_folds) if i != fold_idx])
                        
                        train_mask = np.isin(y_subjects, train_subjects)
                        test_mask = np.isin(y_subjects, test_subjects)
                        
                        train_indices = np.where(train_mask)[0]
                        test_indices = np.where(test_mask)[0]
                        
                        yield train_indices, test_indices
                
                # 执行分类器测试
                classifiers = {
                    'logistic_regression': LogisticRegression(random_state=42, max_iter=1000, C=0.1),
                    'random_forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
                }
                
                # 添加其他分类器（如果可用）
                try:
                    from sklearn.ensemble import GradientBoostingClassifier
                    classifiers['gradient_boosting'] = GradientBoostingClassifier(n_estimators=50, max_depth=3, random_state=42)
                except:
                    pass
                
                try:
                    from sklearn.svm import LinearSVC
                    classifiers['svm_linear'] = LinearSVC(random_state=42, max_iter=1000, C=0.1)
                except:
                    pass
                
                # 执行分类器评估
                classifier_results = {}
                
                for clf_name, clf in classifiers.items():
                    try:
                        print(f"    🔍 测试 {clf_name}...")
                        
                        cv_scores = []
                        for train_idx, test_idx in subject_aware_cv_split(y_subset_remapped, cv_folds):
                            if len(train_idx) > 0 and len(test_idx) > 0:
                                X_train_fold = X_subset[train_idx]
                                y_train_fold = y_subset_remapped[train_idx]
                                X_test_fold = X_subset[test_idx]
                                y_test_fold = y_subset_remapped[test_idx]
                                
                                # 确保训练集包含足够的类别
                                train_classes = np.unique(y_train_fold)
                                test_classes = np.unique(y_test_fold)
                                
                                if len(train_classes) >= 2 and len(test_classes) >= 1:
                                    clf.fit(X_train_fold, y_train_fold)
                                    score = clf.score(X_test_fold, y_test_fold)
                                    cv_scores.append(score)
                        
                        if len(cv_scores) > 0:
                            classifier_results[clf_name] = {
                                'mean_score': np.mean(cv_scores),
                                'std_score': np.std(cv_scores),
                                'scores': cv_scores,
                                'n_folds': len(cv_scores)
                            }
                            print(f"      ✅ {clf_name}: {np.mean(cv_scores):.3f} ± {np.std(cv_scores):.3f} ({len(cv_scores)} folds)")
                        else:
                            print(f"      ⚠️ {clf_name}: 无有效的CV折")
                            
                    except Exception as e:
                        print(f"      ❌ {clf_name}: {e}")
                
                # 计算特征重要性（使用最佳分类器）
                print("    🔍 计算特征重要性...")
                feature_importance = np.zeros(X_subset.shape[1])
                
                try:
                    if 'random_forest' in classifier_results:
                        # 使用随机森林计算特征重要性
                        rf_clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
                        
                        # 为了加速，进一步采样
                        if len(X_subset) > 50000:
                            sample_indices = np.random.choice(len(X_subset), 50000, replace=False)
                            X_importance = X_subset[sample_indices]
                            y_importance = y_subset_remapped[sample_indices]
                        else:
                            X_importance = X_subset
                            y_importance = y_subset_remapped
                        
                        rf_clf.fit(X_importance, y_importance)
                        feature_importance = rf_clf.feature_importances_
                        print(f"      ✅ 特征重要性计算完成")
                    else:
                        print(f"      ⚠️ 无可用分类器计算特征重要性")
                        
                except Exception as e:
                    print(f"      ⚠️ 特征重要性计算失败: {e}")
                
                # 计算整体性能
                if classifier_results:
                    best_clf = max(classifier_results.keys(), key=lambda x: classifier_results[x]['mean_score'])
                    subject_id_accuracy = classifier_results[best_clf]['mean_score']
                    print(f"    ✅ 最佳分类器: {best_clf}")
                else:
                    subject_id_accuracy = 0.0
                    print(f"    ❌ 所有分类器都失败了")
                
                # 保存结果
                self.analysis_results['subject_identification'] = {
                    'accuracy': subject_id_accuracy,
                    'classifier_results': classifier_results,
                    'feature_importance': feature_importance,
                    'valid_subjects': valid_subjects,
                    'n_valid_subjects': len(valid_subjects),
                    'random_baseline': 1.0 / len(valid_subjects),
                    'samples_used': len(X_subset),
                    'cv_strategy': 'subject_aware',
                    'cv_folds_used': cv_folds
                }
                
                print(f"✅ 受试者识别测试完成")
                print(f"  - 识别准确率: {subject_id_accuracy:.3f}")
                print(f"  - 随机基线: {1/len(valid_subjects):.3f}")
                if subject_id_accuracy > 0:
                    improvement = (subject_id_accuracy - 1/len(valid_subjects))/(1/len(valid_subjects))*100
                    print(f"  - 准确率提升: {improvement:.1f}%")
                print(f"  - 使用样本数: {len(X_subset):,}")
                print(f"  - CV策略: 受试者感知 ({cv_folds}折)")
                
            except Exception as e:
                print(f"❌ 受试者识别测试失败: {e}")
                self.analysis_results['subject_identification'] = {
                    'accuracy': 0.0, 
                    'error': str(e),
                    'valid_subjects': valid_subjects,
                    'n_valid_subjects': len(valid_subjects)
                }
                subject_id_accuracy = 0.0
                
        else:
            print("❌ 没有足够样本的受试者进行识别测试")
            self.analysis_results['subject_identification'] = {
                'accuracy': 0.0,
                'n_valid_subjects': 0,
                'error': 'insufficient_subjects'
            }
            subject_id_accuracy = 0.0
        
        # 2.2 脑区分类一致性分析 (保持原有逻辑，略微增强)
        print("\n🔍 2.2 脑区分类一致性分析...")
        
        # 转换one-hot标签到类别索引
        if len(self.data['y_train'].shape) > 1 and self.data['y_train'].shape[1] > 1:
            y_train_classes = np.argmax(self.data['y_train'], axis=1)
        else:
            y_train_classes = self.data['y_train'].flatten()
        
        unique_classes = np.unique(y_train_classes)
        n_classes = len(unique_classes)
        
        print(f"  - 脑区类别数量: {n_classes}")
        print(f"  - 分析前20个脑区的一致性...")
        
        # 计算每个脑区在不同受试者间的一致性
        class_consistency = {}
        
        # 确保受试者ID在这里也是整数
        subjects_train_int = self.data['subjects_train'].astype(int)
        
        for class_id in unique_classes[:20]:  # 只分析前20个类别，避免计算过久
            class_mask = y_train_classes == class_id
            
            if np.sum(class_mask) > 100:  # 确保有足够样本
                class_data = self.data['X_train'][class_mask]
                class_subjects = subjects_train_int[class_mask]
                
                # 计算每个受试者该脑区的均值特征
                subject_means_for_class = []
                subjects_with_class = []
                
                for subject_id in np.unique(class_subjects):
                    subject_class_mask = class_subjects == subject_id
                    if np.sum(subject_class_mask) >= 10:  # 至少10个样本
                        subject_mean = np.mean(class_data[subject_class_mask], axis=0)
                        subject_means_for_class.append(subject_mean)
                        subjects_with_class.append(subject_id)
                
                if len(subject_means_for_class) >= 3:  # 至少3个受试者
                    subject_means_array = np.array(subject_means_for_class)
                    
                    # 计算受试者间的变异系数
                    feature_cv = np.std(subject_means_array, axis=0) / (np.abs(np.mean(subject_means_array, axis=0)) + 1e-8)
                    mean_cv = np.mean(feature_cv)
                    
                    # 计算受试者间距离
                    distances = pdist(subject_means_array)
                    mean_distance = np.mean(distances)
                    
                    class_consistency[class_id] = {
                        'mean_cv': mean_cv,
                        'mean_distance': mean_distance,
                        'n_subjects': len(subjects_with_class),
                        'n_samples': np.sum(class_mask),
                        'subjects_with_class': subjects_with_class
                    }
        
        self.analysis_results['class_consistency'] = class_consistency
        
        if class_consistency:
            avg_consistency = np.mean([info['mean_cv'] for info in class_consistency.values()])
            avg_distance = np.mean([info['mean_distance'] for info in class_consistency.values()])
            
            print(f"✅ 脑区一致性分析完成")
            print(f"  - 分析脑区数量: {len(class_consistency)}")
            print(f"  - 平均变异系数: {avg_consistency:.3f} (越低越一致)")
            print(f"  - 平均受试者间距离: {avg_distance:.3f}")
            
            # 找出最一致和最不一致的脑区
            most_consistent_class = min(class_consistency.keys(), 
                                    key=lambda x: class_consistency[x]['mean_cv'])
            least_consistent_class = max(class_consistency.keys(), 
                                    key=lambda x: class_consistency[x]['mean_cv'])
            
            print(f"  - 最一致脑区: {most_consistent_class} (CV: {class_consistency[most_consistent_class]['mean_cv']:.3f})")
            print(f"  - 最不一致脑区: {least_consistent_class} (CV: {class_consistency[least_consistent_class]['mean_cv']:.3f})")
            
        else:
            print("❌ 脑区一致性分析失败")
            avg_consistency = 1.0
        
        # 2.3 Multi-Subject-Out特定分析 (保持原有逻辑)
        print("\n🔍 2.3 Multi-Subject-Out特定分析...")
        
        multi_subject_analysis = {}
        
        if 'split_strategy' in self.data and self.data['split_strategy'] == 'Multi-Subject-Out':
            # 分析训练集和验证集受试者的差异
            train_subjects = np.unique(self.data['subjects_train'])
            val_subjects = np.unique(self.data['subjects_val'])
            test_subjects = np.unique(self.data['subjects_test'])
            
            print(f"  - 训练集受试者数: {len(train_subjects)} (ID: {sorted(train_subjects)})")
            print(f"  - 验证集受试者数: {len(val_subjects)} (ID: {sorted(val_subjects)})")
            print(f"  - 测试集受试者数: {len(test_subjects)} (ID: {sorted(test_subjects)})")
            
            # 分析受试者分布的代表性
            if 'subject_stats' in self.analysis_results:
                subject_means = self.analysis_results['subject_stats']['means']
                available_subjects = self.data['available_subjects']
                
                # 训练集和验证集受试者的特征分布对比
                train_idx = [i for i, s in enumerate(available_subjects) if s in train_subjects]
                val_idx = [i for i, s in enumerate(available_subjects) if s in val_subjects]
                test_idx = [i for i, s in enumerate(available_subjects) if s in test_subjects]
                
                if len(train_idx) > 0 and len(val_idx) > 0:
                    train_features_mean = np.mean(subject_means[train_idx], axis=0)
                    val_features_mean = np.mean(subject_means[val_idx], axis=0)
                    
                    # 🔧 修复：使用鲁棒的相似性计算
                    try:
                        feature_distribution_similarity = np.corrcoef(train_features_mean, val_features_mean)[0, 1]
                        if not np.isfinite(feature_distribution_similarity):
                            # 备用方法：使用余弦相似性
                            from scipy.spatial.distance import cosine
                            feature_distribution_similarity = 1 - cosine(train_features_mean, val_features_mean)
                    except:
                        feature_distribution_similarity = 0.5  # 默认值
                    
                    # 训练集内部受试者相似性
                    if len(train_idx) > 1:
                        train_internal_distances = pdist(subject_means[train_idx])
                        train_internal_similarity = np.mean(train_internal_distances)
                    else:
                        train_internal_similarity = 0.0
                    
                    # 验证集内部受试者相似性
                    if len(val_idx) > 1:
                        val_internal_distances = pdist(subject_means[val_idx])
                        val_internal_similarity = np.mean(val_internal_distances)
                    else:
                        val_internal_similarity = 0.0
                    
                    # 训练集到验证集的距离
                    train_to_val_distances = []
                    for train_i in train_idx:
                        for val_i in val_idx:
                            distance = np.linalg.norm(subject_means[train_i] - subject_means[val_i])
                            train_to_val_distances.append(distance)
                    
                    avg_train_to_val_distance = np.mean(train_to_val_distances)
                    
                    multi_subject_analysis = {
                        'train_subjects': train_subjects,
                        'val_subjects': val_subjects,
                        'test_subjects': test_subjects,
                        'feature_distribution_similarity': feature_distribution_similarity,
                        'train_internal_similarity': train_internal_similarity,
                        'val_internal_similarity': val_internal_similarity,
                        'train_to_val_distance': avg_train_to_val_distance,
                        'train_features_mean': train_features_mean,
                        'val_features_mean': val_features_mean
                    }
                    
                    print(f"  - 训练/验证集特征分布相似性: {feature_distribution_similarity:.3f}")
                    print(f"  - 训练集内部平均距离: {train_internal_similarity:.3f}")
                    print(f"  - 验证集内部平均距离: {val_internal_similarity:.3f}")
                    print(f"  - 训练集到验证集平均距离: {avg_train_to_val_distance:.3f}")
                    
                    # 评估跨受试者泛化难度
                    if avg_train_to_val_distance > train_internal_similarity * 1.5:
                        print("  ⚠️ 验证集受试者与训练集差异较大，跨受试者泛化可能困难")
                    elif avg_train_to_val_distance < train_internal_similarity * 1.2:
                        print("  ✅ 验证集受试者与训练集相似，跨受试者泛化相对容易")
                    else:
                        print("  📊 验证集受试者与训练集差异适中")
        
        self.analysis_results['multi_subject_out_analysis'] = multi_subject_analysis
        
        # Phase 2 决策得分计算
        n_valid_subjects = len(valid_subjects) if len(valid_subjects) > 0 else 1
        random_baseline = 1 / n_valid_subjects
        separability_score = max(0, min(1, (subject_id_accuracy - random_baseline) / (1 - random_baseline + 1e-8)))
        
        if class_consistency:
            consistency_score = max(0, min(1, 1 / (1 + avg_consistency)))  # 变异系数越低，得分越高
        else:
            consistency_score = 0.5
        
        # Multi-Subject-Out特定得分
        multi_subject_score = 0.5  # 默认值
        if multi_subject_analysis and 'feature_distribution_similarity' in multi_subject_analysis:
            # 相似性越高，跨受试者学习越容易
            similarity = multi_subject_analysis['feature_distribution_similarity']
            if np.isfinite(similarity):
                multi_subject_score = max(0, min(1, (similarity + 1) / 2))  # 将[-1,1]映射到[0,1]
        
        self.decision_scores['phase2'] = {
            'subject_separability': separability_score,
            'class_consistency': consistency_score,
            'identification_accuracy': subject_id_accuracy,
            'feature_competition_risk': separability_score,
            'multi_subject_generalization': multi_subject_score,
            'random_baseline': random_baseline
        }
        
        print(f"\n📈 Phase 2 决策指标:")
        print(f"  - 受试者可分离性: {separability_score:.3f}")
        print(f"  - 脑区分类一致性: {consistency_score:.3f}")
        print(f"  - 受试者识别准确率: {subject_id_accuracy:.3f}")
        print(f"  - 特征竞争风险: {separability_score:.3f}")
        print(f"  - 跨受试者泛化性: {multi_subject_score:.3f}")
        print(f"  - 随机基线: {random_baseline:.3f}")
        
        print(f"\n✅ Phase 2 分析完成")
    
    def phase3_embedding_adaptability_analysis(self):
        """
        Phase 3: Embedding适配性评估 - 增强版
        
        3.1 多种降维方法对比分析
        3.2 高维空间直接分析
        3.3 特征分组分析
        3.4 综合适配性评估
        """
        print("\n" + "="*80)
        print("📊 Phase 3: Embedding适配性评估 (增强版)")
        print("="*80)
        
        subject_means = self.analysis_results['subject_stats']['means']
        
        # 3.1 多种降维方法对比分析
        print("\n🔍 3.1 多种降维方法对比分析...")
        dimensionality_results = self._comprehensive_dimensionality_analysis(subject_means)
        
        # 3.2 高维空间直接分析
        print("\n🔍 3.2 高维空间直接分析...")
        high_dim_results = self._high_dimensional_direct_analysis(subject_means)
        
        # 3.3 特征分组分析
        print("\n🔍 3.3 特征分组分析...")
        group_results = self._feature_group_analysis(subject_means)
        
        # 3.4 样本量充足性评估 (保持原有逻辑)
        print("\n🔍 3.4 样本量充足性评估...")
        sample_adequacy = self._sample_adequacy_assessment()
        
        # 3.5 综合适配性评估和决策
        print("\n🔍 3.5 综合适配性评估...")
        comprehensive_assessment = self._comprehensive_embedding_assessment(
            dimensionality_results, high_dim_results, group_results, sample_adequacy
        )
        
        # 存储所有结果
        self.analysis_results['enhanced_dimensionality'] = {
            'dimensionality_comparison': dimensionality_results,
            'high_dimensional_analysis': high_dim_results,
            'feature_group_analysis': group_results,
            'sample_adequacy': sample_adequacy,
            'comprehensive_assessment': comprehensive_assessment
        }
        
        # 更新决策得分 (兼容原有逻辑)
        self.decision_scores['phase3'] = {
            'linearity': comprehensive_assessment['linearity_score'],
            'clustering_quality': comprehensive_assessment['clustering_score'],
            'sample_adequacy': sample_adequacy['adequacy_score'],
            'embedding_feasibility': comprehensive_assessment['overall_feasibility'],
            'intrinsic_dimensionality': comprehensive_assessment['intrinsic_dim_score'],
            'high_dim_performance': comprehensive_assessment['high_dim_score']
        }
        
        print(f"✅ 增强版适配性分析完成")
        print(f"📈 Phase 3 增强决策指标:")
        print(f"  - 线性度: {self.decision_scores['phase3']['linearity']:.3f}")
        print(f"  - 聚类质量: {self.decision_scores['phase3']['clustering_quality']:.3f}")
        print(f"  - 样本充足性: {self.decision_scores['phase3']['sample_adequacy']:.3f}")
        print(f"  - 内在维度得分: {self.decision_scores['phase3']['intrinsic_dimensionality']:.3f}")
        print(f"  - 高维性能得分: {self.decision_scores['phase3']['high_dim_performance']:.3f}")
        print(f"  - 整体可行性: {self.decision_scores['phase3']['embedding_feasibility']:.3f}")

    def _comprehensive_dimensionality_analysis(self, subject_means):
        """多种降维方法对比分析"""
        results = {}
        
        # 1. PCA (线性)
        pca = PCA()
        pca_result = pca.fit_transform(subject_means)
        pca_2d = PCA(n_components=2).fit_transform(subject_means)
        
        results['pca'] = {
            'full_result': pca_result,
            '2d_result': pca_2d,
            'explained_variance': pca.explained_variance_ratio_,
            'cumulative_variance': np.cumsum(pca.explained_variance_ratio_),
            'method_type': 'linear'
        }
        
        # 2. Kernel PCA (非线性)
        try:
            from sklearn.decomposition import KernelPCA
            
            # RBF kernel
            kpca_rbf = KernelPCA(n_components=min(10, len(subject_means)-1), 
                            kernel='rbf', gamma=0.1, random_state=42)
            kpca_rbf_result = kpca_rbf.fit_transform(subject_means)
            
            kpca_rbf_2d = KernelPCA(n_components=2, kernel='rbf', gamma=0.1, random_state=42)
            kpca_rbf_2d_result = kpca_rbf_2d.fit_transform(subject_means)
            
            # Polynomial kernel
            kpca_poly = KernelPCA(n_components=2, kernel='poly', degree=2, random_state=42)
            kpca_poly_2d_result = kpca_poly.fit_transform(subject_means)
            
            results['kernel_pca'] = {
                'rbf_result': kpca_rbf_result,
                'rbf_2d_result': kpca_rbf_2d_result,
                'poly_2d_result': kpca_poly_2d_result,
                'method_type': 'nonlinear_kernel'
            }
            
            print(f"    ✅ Kernel PCA分析完成")
            
        except Exception as e:
            print(f"    ⚠️ Kernel PCA分析失败: {e}")
            results['kernel_pca'] = None
        
        # 3. t-SNE (非线性流形)
        try:
            perplexity = min(30, len(subject_means)-1)
            tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity, 
                    learning_rate='auto', init='random')
            tsne_result = tsne.fit_transform(subject_means)
            
            results['tsne'] = {
                '2d_result': tsne_result,
                'perplexity': perplexity,
                'method_type': 'nonlinear_manifold'
            }
            
            print(f"    ✅ t-SNE分析完成 (perplexity={perplexity})")
            
        except Exception as e:
            print(f"    ⚠️ t-SNE分析失败: {e}")
            results['tsne'] = None
        
        # 4. UMAP (更好的非线性降维)
        try:
            import umap
            umap_reducer = umap.UMAP(n_components=2, random_state=42, 
                                n_neighbors=min(15, len(subject_means)-1))
            umap_result = umap_reducer.fit_transform(subject_means)
            
            results['umap'] = {
                '2d_result': umap_result,
                'method_type': 'nonlinear_manifold'
            }
            
            print(f"    ✅ UMAP分析完成")
            
        except ImportError:
            print(f"    ⚠️ UMAP不可用，请安装: pip install umap-learn")
            results['umap'] = None
        except Exception as e:
            print(f"    ⚠️ UMAP分析失败: {e}")
            results['umap'] = None
        
        # 5. 比较不同方法的结构保持能力
        structure_preservation = self._compare_structure_preservation(subject_means, results)
        results['structure_preservation'] = structure_preservation
        
        return results

    def _high_dimensional_direct_analysis(self, subject_means):
        """高维空间直接分析"""
        results = {}
        
        # 1. 高维距离分析
        print("    🔍 计算多种距离度量...")
        distance_metrics = {
            'euclidean': 'euclidean',
            'cosine': 'cosine', 
            'manhattan': 'manhattan',
            'chebyshev': 'chebyshev'
        }
        
        distance_matrices = {}
        for metric_name, metric in distance_metrics.items():
            try:
                distances = squareform(pdist(subject_means, metric=metric))
                distance_matrices[metric_name] = distances
            except Exception as e:
                print(f"      ⚠️ {metric_name}距离计算失败: {e}")
        
        results['distance_matrices'] = distance_matrices
        
        # 2. 高维聚类分析
        print("    🔍 高维聚类分析...")
        clustering_results = {}
        
        # DBSCAN (密度聚类，适合高维)
        from sklearn.cluster import DBSCAN
        eps_values = [0.1, 0.5, 1.0, 2.0, 5.0]
        
        for eps in eps_values:
            try:
                dbscan = DBSCAN(eps=eps, min_samples=max(2, len(subject_means)//10))
                labels = dbscan.fit_predict(subject_means)
                n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
                
                if n_clusters > 1:
                    clustering_results[f'dbscan_eps_{eps}'] = {
                        'labels': labels,
                        'n_clusters': n_clusters,
                        'n_noise': list(labels).count(-1)
                    }
            except Exception as e:
                print(f"      ⚠️ DBSCAN (eps={eps}) 失败: {e}")
        
        results['clustering_results'] = clustering_results
        
        # 3. 高维可分离性测试
        print("    🔍 高维分类器性能测试...")
        separability_scores = self._test_high_dim_classifiers(subject_means)
        results['separability_scores'] = separability_scores
        
        # 4. 内在维度估计
        print("    🔍 估计内在维度...")
        intrinsic_dim = self._estimate_intrinsic_dimensionality(subject_means)
        results['intrinsic_dimensionality'] = intrinsic_dim
        
        # 5. 距离集中度分析（高维诅咒检测）
        if 'euclidean' in distance_matrices:
            concentration = self._measure_distance_concentration(distance_matrices['euclidean'])
            results['distance_concentration'] = concentration
        
        # 6. 特征重要性分析（无降维）
        print("    🔍 计算特征重要性...")
        feature_importance = self._compute_feature_importance_high_dim()
        results['feature_importance'] = feature_importance
        
        print(f"    ✅ 高维分析完成")
        print(f"      - 估计内在维度: {intrinsic_dim:.1f}")
        print(f"      - 有效聚类方法: {len(clustering_results)}")
        print(f"      - 测试分类器: {len(separability_scores)}")
        
        return results

    def _feature_group_analysis(self, subject_means):
        """特征分组分析"""
        
        # 根据341维特征的典型组成定义分组
        # 需要根据你的具体数据调整这些索引
        feature_groups = {
            'qti_params': list(range(0, 15)),           # QTI参数 (15个)
            'raw_b_tensors': list(range(15, 225)),      # 原始b-tensor值 (210个)
            'cest_params': list(range(225, 229)),       # CEST参数 (4个)
            'z_spectrum': list(range(229, 341))         # Z-spectrum值 (112个)
        }
        
        group_analysis = {}
        
        for group_name, feature_indices in feature_groups.items():
            if len(feature_indices) == 0:
                continue
                
            print(f"    🔍 分析 {group_name} ({len(feature_indices)} 特征)...")
            
            try:
                # 提取该组特征
                group_data = subject_means[:, feature_indices]
                
                if len(feature_indices) >= 2:
                    # PCA分析
                    pca = PCA()
                    pca_result = pca.fit_transform(group_data)
                    
                    # 计算该组特征的受试者间差异
                    group_distances = squareform(pdist(group_data))
                    
                    # 层次聚类
                    linkage_matrix = linkage(group_data, method='ward')
                    
                    # 该组特征的可分离性
                    group_separability = 0
                    if len(feature_indices) >= 3:
                        try:
                            rf = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
                            # 简化的分类任务：用该组特征预测受试者分组
                            subject_labels = np.arange(len(subject_means))
                            scores = cross_val_score(rf, group_data, subject_labels, cv=min(3, len(subject_means)))
                            group_separability = np.mean(scores)
                        except:
                            pass
                    
                    group_analysis[group_name] = {
                        'pca_explained_variance': pca.explained_variance_ratio_,
                        'pca_cumulative_variance': np.cumsum(pca.explained_variance_ratio_),
                        'variance_in_3pc': np.sum(pca.explained_variance_ratio_[:3]) if len(pca.explained_variance_ratio_) >= 3 else np.sum(pca.explained_variance_ratio_),
                        'mean_distance': np.mean(group_distances),
                        'distance_std': np.std(group_distances),
                        'coefficient_of_variation': np.std(group_distances) / (np.mean(group_distances) + 1e-8),
                        'linkage_matrix': linkage_matrix,
                        'feature_count': len(feature_indices),
                        'separability_score': group_separability,
                        'feature_range': (min(feature_indices), max(feature_indices))
                    }
                    
                    print(f"      ✅ {group_name}: 前3PC解释方差 {group_analysis[group_name]['variance_in_3pc']:.3f}")
                    
            except Exception as e:
                print(f"      ⚠️ {group_name} 分析失败: {e}")
                group_analysis[group_name] = {'error': str(e)}
        
        return group_analysis

    def _test_high_dim_classifiers(self, subject_means):
        """测试多种高维分类器的性能"""
        
        # 准备分类数据
        classifiers = {
            'random_forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
            'gradient_boosting': None,  # 稍后初始化
            'svm_linear': None,  # 稍后初始化
            'logistic_regression': LogisticRegression(random_state=42, max_iter=1000, C=0.1)
        }
        
        # 动态初始化需要参数的分类器
        try:
            from sklearn.ensemble import GradientBoostingClassifier
            classifiers['gradient_boosting'] = GradientBoostingClassifier(n_estimators=50, max_depth=3, random_state=42)
        except:
            pass
        
        try:
            from sklearn.svm import LinearSVC
            classifiers['svm_linear'] = LinearSVC(random_state=42, max_iter=1000, C=0.1)
        except:
            pass
        
        separability_scores = {}
        
        # 使用受试者均值进行分类测试
        subject_labels = np.arange(len(subject_means))
        
        for clf_name, clf in classifiers.items():
            if clf is None:
                continue
                
            try:
                if len(subject_means) >= 5:  # 确保有足够样本进行交叉验证
                    cv_folds = min(5, len(subject_means))
                    scores = cross_val_score(clf, subject_means, subject_labels, cv=cv_folds, scoring='accuracy')
                    separability_scores[clf_name] = {
                        'mean_score': np.mean(scores),
                        'std_score': np.std(scores),
                        'scores': scores
                    }
                else:
                    # 样本太少，使用训练集准确率
                    clf.fit(subject_means, subject_labels)
                    train_score = clf.score(subject_means, subject_labels)
                    separability_scores[clf_name] = {
                        'mean_score': train_score,
                        'std_score': 0.0,
                        'scores': [train_score]
                    }
                    
            except Exception as e:
                print(f"      ⚠️ {clf_name} 测试失败: {e}")
        
        return separability_scores

    def _estimate_intrinsic_dimensionality(self, data):
        """估计数据的内在维度"""
        try:
            from sklearn.neighbors import NearestNeighbors
            
            k = min(10, len(data) - 1)
            if k < 2:
                return data.shape[1]
                
            nbrs = NearestNeighbors(n_neighbors=k+1).fit(data)
            distances, indices = nbrs.kneighbors(data)
            
            # Levina-Bickel估计器
            distances = distances[:, 1:]  # 排除自身距离
            
            # 计算最远邻居与最近邻居的距离比值
            ratios = distances[:, -1] / (distances[:, 0] + 1e-10)
            
            # 过滤掉异常值
            ratios = ratios[ratios > 1]
            ratios = ratios[ratios < 1000]  # 移除极端值
            
            if len(ratios) > 0:
                log_ratios = np.log(ratios)
                intrinsic_dim = k / np.mean(log_ratios)
                return min(max(1, intrinsic_dim), data.shape[1])  # 限制在合理范围内
            else:
                return data.shape[1]
                
        except Exception as e:
            print(f"      ⚠️ 内在维度估计失败: {e}")
            return data.shape[1]

    def _measure_distance_concentration(self, distance_matrix):
        """测量距离集中度（高维诅咒指标）"""
        try:
            # 获取上三角矩阵的距离值
            upper_triangle = distance_matrix[np.triu_indices_from(distance_matrix, k=1)]
            upper_triangle = upper_triangle[upper_triangle > 0]
            
            if len(upper_triangle) > 0:
                # 距离集中度：标准差/均值
                concentration = np.std(upper_triangle) / (np.mean(upper_triangle) + 1e-10)
                return concentration
            else:
                return 1.0
        except:
            return 1.0

    def _compute_feature_importance_high_dim(self):
        """计算高维特征重要性"""
        feature_importance = np.zeros(self.data['X_train'].shape[1])
        
        try:
            # 使用方差分析计算每个特征的受试者间差异
            for feat_idx in range(self.data['X_train'].shape[1]):
                feature_values_by_subject = []
                
                for subject_id in self.data['available_subjects']:
                    subject_mask = self.data['subjects_train'] == subject_id
                    if np.sum(subject_mask) >= 10:  # 确保足够样本
                        subject_feature_values = self.data['X_train'][subject_mask, feat_idx]
                        feature_values_by_subject.append(np.mean(subject_feature_values))
                
                if len(feature_values_by_subject) >= 3:
                    # 使用变异系数作为重要性指标
                    mean_val = np.mean(feature_values_by_subject)
                    std_val = np.std(feature_values_by_subject)
                    cv = std_val / (abs(mean_val) + 1e-8)
                    feature_importance[feat_idx] = cv
            
            # 归一化到[0,1]
            if np.max(feature_importance) > 0:
                feature_importance = feature_importance / np.max(feature_importance)
                
        except Exception as e:
            print(f"      ⚠️ 特征重要性计算失败: {e}")
        
        return feature_importance

    def _compare_structure_preservation(self, original_data, dimensionality_results):
        """比较不同降维方法的结构保持能力"""
        
        # 计算原始高维空间的距离矩阵
        original_distances = squareform(pdist(original_data))
        
        preservation_scores = {}
        
        # 对每种降维方法计算结构保持能力
        for method_name, method_data in dimensionality_results.items():
            if method_data is None or method_name == 'structure_preservation':
                continue
                
            try:
                # 获取2D结果
                if '2d_result' in method_data:
                    reduced_data = method_data['2d_result']
                elif method_name == 'pca' and 'full_result' in method_data:
                    reduced_data = method_data['full_result'][:, :2]
                else:
                    continue
                
                # 计算降维后的距离矩阵
                reduced_distances = squareform(pdist(reduced_data))
                
                # 计算Spearman相关系数（排序保持）
                from scipy.stats import spearmanr
                correlation, p_value = spearmanr(
                    original_distances.flatten(), 
                    reduced_distances.flatten()
                )
                
                preservation_scores[method_name] = {
                    'spearman_correlation': correlation,
                    'p_value': p_value
                }
                
            except Exception as e:
                print(f"      ⚠️ {method_name} 结构保持分析失败: {e}")
        
        return preservation_scores

    def _sample_adequacy_assessment(self):
        """样本量充足性评估（保持原有逻辑）"""
        sample_counts = self.analysis_results['subject_stats']['sample_counts']
        total_samples = np.sum(sample_counts)
        n_subjects = len(sample_counts)
        avg_samples_per_subject = np.mean(sample_counts)
        min_samples_per_subject = np.min(sample_counts)
        
        # 评估样本量是否充足
        sample_adequacy_score = 0
        
        # 评估每个受试者的样本量
        if avg_samples_per_subject >= 50000:
            sample_adequacy_score += 0.4
        elif avg_samples_per_subject >= 10000:
            sample_adequacy_score += 0.2
        
        # 评估受试者数量
        if n_subjects >= 30:
            sample_adequacy_score += 0.3
        elif n_subjects >= 20:
            sample_adequacy_score += 0.2
        
        # 评估总样本量
        if total_samples >= 1000000:
            sample_adequacy_score += 0.3
        elif total_samples >= 500000:
            sample_adequacy_score += 0.2
        
        return {
            'total_samples': total_samples,
            'n_subjects': n_subjects,
            'avg_samples_per_subject': avg_samples_per_subject,
            'min_samples_per_subject': min_samples_per_subject,
            'adequacy_score': sample_adequacy_score
        }

    def _comprehensive_embedding_assessment(self, dimensionality_results, high_dim_results, group_results, sample_adequacy):
        """综合embedding适配性评估"""
        
        assessment = {
            'linearity_score': 0.0,
            'clustering_score': 0.0,
            'intrinsic_dim_score': 0.0,
            'high_dim_score': 0.0,
            'overall_feasibility': 0.0,
            'recommendations': {}
        }
        
        # 1. 线性度评估
        if 'pca' in dimensionality_results:
            pca_3pc_variance = np.sum(dimensionality_results['pca']['explained_variance'][:3])
            assessment['linearity_score'] = min(1.0, pca_3pc_variance * 1.25)  # 放大权重
        
        # 结构保持能力
        if 'structure_preservation' in dimensionality_results:
            preservation_scores = []
            for method, scores in dimensionality_results['structure_preservation'].items():
                if 'spearman_correlation' in scores:
                    preservation_scores.append(abs(scores['spearman_correlation']))
            
            if preservation_scores:
                avg_preservation = np.mean(preservation_scores)
                assessment['linearity_score'] = (assessment['linearity_score'] + avg_preservation) / 2
        
        # 2. 聚类质量评估
        clustering_scores = []
        if 'clustering_results' in high_dim_results:
            # 评估DBSCAN结果
            for clustering_name, clustering_info in high_dim_results['clustering_results'].items():
                n_clusters = clustering_info['n_clusters']
                n_noise = clustering_info['n_noise']
                total_points = len(clustering_info['labels'])
                
                if n_clusters > 1 and n_noise / total_points < 0.5:  # 不超过50%噪点
                    cluster_quality = n_clusters / total_points  # 简化的质量评估
                    clustering_scores.append(min(1.0, cluster_quality * 10))
        
        if clustering_scores:
            assessment['clustering_score'] = np.mean(clustering_scores)
        
        # 3. 内在维度评估
        if 'intrinsic_dimensionality' in high_dim_results:
            intrinsic_dim = high_dim_results['intrinsic_dimensionality']
            # 内在维度越低，embedding越有效
            if intrinsic_dim <= 20:
                assessment['intrinsic_dim_score'] = 1.0
            elif intrinsic_dim <= 50:
                assessment['intrinsic_dim_score'] = 0.8
            elif intrinsic_dim <= 100:
                assessment['intrinsic_dim_score'] = 0.6
            else:
                assessment['intrinsic_dim_score'] = 0.3
        
        # 4. 高维性能评估
        if 'separability_scores' in high_dim_results:
            high_dim_scores = []
            for clf_name, clf_results in high_dim_results['separability_scores'].items():
                high_dim_scores.append(clf_results['mean_score'])
            
            if high_dim_scores:
                avg_high_dim_performance = np.mean(high_dim_scores)
                # 转换为0-1分数，考虑随机基线
                n_subjects = len(self.data['available_subjects'])
                random_baseline = 1.0 / n_subjects
                normalized_score = (avg_high_dim_performance - random_baseline) / (1 - random_baseline)
                assessment['high_dim_score'] = max(0, min(1, normalized_score))
        
        # 5. 综合可行性评估
        weights = {
            'linearity': 0.25,
            'clustering': 0.15,
            'intrinsic_dim': 0.25,
            'high_dim': 0.25,
            'sample_adequacy': 0.10
        }
        
        assessment['overall_feasibility'] = (
            assessment['linearity_score'] * weights['linearity'] +
            assessment['clustering_score'] * weights['clustering'] +
            assessment['intrinsic_dim_score'] * weights['intrinsic_dim'] +
            assessment['high_dim_score'] * weights['high_dim'] +
            sample_adequacy['adequacy_score'] * weights['sample_adequacy']
        )
        
        # 6. 生成具体建议
        recommendations = {}
        
        # Embedding维度建议
        if 'intrinsic_dimensionality' in high_dim_results:
            intrinsic_dim = high_dim_results['intrinsic_dimensionality']
            if assessment['linearity_score'] > 0.8:
                recommendations['embedding_dim'] = max(16, min(64, int(intrinsic_dim * 1.5)))
                recommendations['embedding_type'] = 'linear'
            elif assessment['linearity_score'] > 0.6:
                recommendations['embedding_dim'] = max(32, min(128, int(intrinsic_dim * 2)))
                recommendations['embedding_type'] = 'mixed'
            else:
                recommendations['embedding_dim'] = max(64, min(256, int(intrinsic_dim * 3)))
                recommendations['embedding_type'] = 'nonlinear'
        else:
            recommendations['embedding_dim'] = 64
            recommendations['embedding_type'] = 'linear'
        
        # 特征选择建议
        if 'feature_importance' in high_dim_results:
            feature_importance = high_dim_results['feature_importance']
            high_importance_features = np.sum(feature_importance > 0.5)
            total_features = len(feature_importance)
            
            if high_importance_features / total_features < 0.3:
                recommendations['feature_selection'] = True
                recommendations['important_features_ratio'] = high_importance_features / total_features
            else:
                recommendations['feature_selection'] = False
        
        # 分组建议
        if group_results:
            group_variances = {}
            for group_name, group_info in group_results.items():
                if 'variance_in_3pc' in group_info:
                    group_variances[group_name] = group_info['variance_in_3pc']
            
            if group_variances:
                best_group = max(group_variances, key=group_variances.get)
                recommendations['best_feature_group'] = best_group
                recommendations['group_variances'] = group_variances
        
        assessment['recommendations'] = recommendations
        
        print(f"    ✅ 综合评估完成")
        print(f"      - 线性度得分: {assessment['linearity_score']:.3f}")
        print(f"      - 内在维度得分: {assessment['intrinsic_dim_score']:.3f}")
        print(f"      - 高维性能得分: {assessment['high_dim_score']:.3f}")
        print(f"      - 整体可行性: {assessment['overall_feasibility']:.3f}")
        
        return assessment
    
    def generate_visualizations(self):
        """生成所有分析结果的可视化图表 - 增强版"""
        print("\n" + "="*80)
        print("📊 生成增强版可视化图表")
        print("="*80)
        
        # 创建更大的图布局 (4x5 = 20个子图)
        fig, axes = plt.subplots(4, 5, figsize=(25, 20))
        
        # 原有的可视化 (前11个)
        self._generate_basic_visualizations(axes)
        
        # 新增的增强版可视化 (后9个)
        self._generate_enhanced_visualizations(axes)
        
        plt.tight_layout(pad=3.0)
        viz_path = os.path.join(self.save_path, 'visualizations', 'comprehensive_enhanced_analysis.png')
        plt.savefig(viz_path, dpi=300, bbox_inches='tight')
        plt.show()
        
        # 生成专门的降维对比图
        self._generate_dimensionality_comparison_plot()
        
        # 生成特征分组分析图
        self._generate_feature_group_plot()
        
        print(f"✅ 增强版可视化图表已保存")



    def _generate_basic_visualizations(self, axes):
        """生成所有分析结果的可视化图表"""
        print("\n" + "="*80)
        print("📊 生成可视化图表")
        print("="*80)
        
        # 创建大图
        fig, axes = plt.subplots(3, 4, figsize=(20, 15))
        
        # 1. 受试者相似性热图
        ax = axes[0, 0]
        correlation_matrix = self.analysis_results['subject_similarity']['correlation_matrix']
        im = ax.imshow(correlation_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
        ax.set_title('受试者间相关性矩阵', fontweight='bold')
        ax.set_xlabel('受试者 ID')
        ax.set_ylabel('受试者 ID')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        
        # 2. 层次聚类树状图
        ax = axes[0, 1]
        linkage_matrix = self.analysis_results['subject_similarity']['linkage_matrix']
        dendrogram(linkage_matrix, ax=ax, leaf_rotation=90)
        ax.set_title('受试者层次聚类', fontweight='bold')
        ax.set_xlabel('受试者 ID')
        ax.set_ylabel('距离')
        
        # 3. 特征变异分布
        ax = axes[0, 2]
        f_stats = self.analysis_results['feature_variation']['f_stats']
        ax.hist(f_stats, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
        ax.axvline(np.percentile(f_stats, 90), color='red', linestyle='--', 
                  label=f'90th percentile: {np.percentile(f_stats, 90):.2f}')
        ax.axvline(np.percentile(f_stats, 10), color='green', linestyle='--',
                  label=f'10th percentile: {np.percentile(f_stats, 10):.2f}')
        ax.set_title('特征受试者间变异分布', fontweight='bold')
        ax.set_xlabel('标准化F统计量')
        ax.set_ylabel('特征数量')
        ax.legend()
        
        # 4. PCA累积方差解释
        ax = axes[0, 3]
        cumulative_variance = self.analysis_results['feature_variation']['pca_cumulative_variance']
        ax.plot(range(1, min(21, len(cumulative_variance)+1)), 
               cumulative_variance[:20], 'o-', linewidth=2, markersize=6)
        ax.axhline(0.8, color='red', linestyle='--', label='80%解释阈值')
        ax.axhline(0.9, color='orange', linestyle='--', label='90%解释阈值')
        ax.set_title('PCA累积方差解释', fontweight='bold')
        ax.set_xlabel('主成分数量')
        ax.set_ylabel('累积方差解释比例')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # 5. 受试者识别准确率
        ax = axes[1, 0]
        if 'subject_identification' in self.analysis_results:
            cv_scores = self.analysis_results['subject_identification'].get('cv_scores', [0])
            accuracy = self.analysis_results['subject_identification'].get('accuracy', 0)
            n_subjects = self.analysis_results['subject_identification'].get('n_valid_subjects', 1)
            random_baseline = 1 / n_subjects
            
            ax.bar(['随机基线', '逻辑回归'], [random_baseline, accuracy], 
                  color=['gray', 'lightcoral'])
            ax.set_title('受试者识别准确率对比', fontweight='bold')
            ax.set_ylabel('准确率')
            
            # 添加数值标签
            ax.text(0, random_baseline + 0.01, f'{random_baseline:.3f}', 
                   ha='center', va='bottom')
            ax.text(1, accuracy + 0.01, f'{accuracy:.3f}', 
                   ha='center', va='bottom')
        
        # 6. 特征重要性（Top 20）
        ax = axes[1, 1]
        if 'subject_identification' in self.analysis_results and 'feature_importance' in self.analysis_results['subject_identification']:
            feature_importance = self.analysis_results['subject_identification']['feature_importance']
            top_features = np.argsort(feature_importance)[-20:]
            ax.barh(range(20), feature_importance[top_features], color='lightgreen')
            ax.set_title('Top 20 受试者识别特征重要性', fontweight='bold')
            ax.set_xlabel('重要性得分')
            ax.set_ylabel('特征索引')
            ax.set_yticks(range(20))
            ax.set_yticklabels(top_features)
        
        # 7. PCA vs t-SNE受试者分布对比
        ax = axes[1, 2]
        if 'linearity_analysis' in self.analysis_results:
            pca_2d = self.analysis_results['linearity_analysis']['pca_2d']
            scatter = ax.scatter(pca_2d[:, 0], pca_2d[:, 1], 
                               c=range(len(pca_2d)), cmap='viridis', s=50, alpha=0.7)
            ax.set_title('PCA - 受试者2D分布', fontweight='bold')
            ax.set_xlabel('PC1')
            ax.set_ylabel('PC2')
            plt.colorbar(scatter, ax=ax, fraction=0.046, pad=0.04)
        
        ax = axes[1, 3]
        if 'linearity_analysis' in self.analysis_results:
            tsne_2d = self.analysis_results['linearity_analysis']['tsne_2d']
            scatter = ax.scatter(tsne_2d[:, 0], tsne_2d[:, 1], 
                               c=range(len(tsne_2d)), cmap='viridis', s=50, alpha=0.7)
            ax.set_title('t-SNE - 受试者2D分布', fontweight='bold')
            ax.set_xlabel('t-SNE 1')
            ax.set_ylabel('t-SNE 2')
            plt.colorbar(scatter, ax=ax, fraction=0.046, pad=0.04)
        
        # 8. 聚类质量评估
        ax = axes[2, 0]
        if 'linearity_analysis' in self.analysis_results and 'silhouette_scores' in self.analysis_results['linearity_analysis']:
            silhouette_scores = self.analysis_results['linearity_analysis']['silhouette_scores']
            n_clusters_range = range(2, 2 + len(silhouette_scores))
            ax.plot(n_clusters_range, silhouette_scores, 'o-', linewidth=2, markersize=8)
            best_n = self.analysis_results['linearity_analysis']['best_n_clusters']
            best_score = self.analysis_results['linearity_analysis']['best_silhouette']
            ax.axvline(best_n, color='red', linestyle='--', 
                      label=f'最优聚类数: {best_n}')
            ax.set_title('聚类质量评估 (轮廓系数)', fontweight='bold')
            ax.set_xlabel('聚类数量')
            ax.set_ylabel('轮廓系数')
            ax.legend()
            ax.grid(True, alpha=0.3)
        
        # 9. 脑区一致性分析
        ax = axes[2, 1]
        if 'class_consistency' in self.analysis_results and self.analysis_results['class_consistency']:
            class_consistency = self.analysis_results['class_consistency']
            class_ids = list(class_consistency.keys())
            cv_values = [class_consistency[cid]['mean_cv'] for cid in class_ids]
            
            ax.bar(range(len(class_ids)), cv_values, color='lightblue', alpha=0.7)
            ax.set_title('脑区受试者间一致性', fontweight='bold')
            ax.set_xlabel('脑区 ID')
            ax.set_ylabel('变异系数 (越低越一致)')
            ax.set_xticks(range(len(class_ids)))
            ax.set_xticklabels(class_ids, rotation=45)
        
        # 10. 样本量分布
        ax = axes[2, 2]
        sample_counts = self.analysis_results['subject_stats']['sample_counts']
        ax.hist(sample_counts, bins=15, alpha=0.7, color='lightcoral', edgecolor='black')
        ax.axvline(np.mean(sample_counts), color='red', linestyle='--',
                  label=f'平均: {np.mean(sample_counts):.0f}')
        ax.axvline(np.median(sample_counts), color='green', linestyle='--',
                  label=f'中位数: {np.median(sample_counts):.0f}')
        ax.set_title('受试者样本量分布', fontweight='bold')
        ax.set_xlabel('样本量')
        ax.set_ylabel('受试者数量')
        ax.legend()
        
        # 11. 综合决策雷达图
        ax = axes[2, 3]
        categories = ['差异显著性', '模式线性度', '受试者相似性', '特征异质性', 
                     '可分离性', '类别一致性', '样本充足性']
        
        scores = [
            self.decision_scores['phase1']['difference_significance'],
            self.decision_scores['phase1']['pattern_linearity'], 
            self.decision_scores['phase1']['subject_similarity'],
            self.decision_scores['phase1']['feature_heterogeneity'],
            self.decision_scores['phase2']['subject_separability'],
            self.decision_scores['phase2']['class_consistency'],
            self.decision_scores['phase3']['sample_adequacy']
        ]
        
        # 雷达图
        angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False)
        scores_plot = scores + [scores[0]]  # 闭合图形
        angles_plot = np.concatenate((angles, [angles[0]]))
        
        ax.plot(angles_plot, scores_plot, 'o-', linewidth=2, color='blue')
        ax.fill(angles_plot, scores_plot, alpha=0.25, color='blue')
        ax.set_xticks(angles)
        ax.set_xticklabels(categories, fontsize=10)
        ax.set_ylim(0, 1)
        ax.set_title('Subject Embedding 可行性雷达图', fontweight='bold')
        ax.grid(True)
        
        plt.tight_layout()
        viz_path = os.path.join(self.save_path, 'visualizations', 'comprehensive_analysis.png')
        plt.savefig(viz_path, dpi=300, bbox_inches='tight')
        plt.show()
        
        print(f"✅ 可视化图表已保存: {viz_path}")
    
    
    def _generate_enhanced_visualizations(self, axes):
        """生成新增的增强版可视化"""
        
        if 'enhanced_dimensionality' not in self.analysis_results:
            return
        
        enhanced_results = self.analysis_results['enhanced_dimensionality']
        
        # 12. 降维方法对比 (2,4)
        ax = axes[2, 4]
        if 'dimensionality_comparison' in enhanced_results:
            dim_results = enhanced_results['dimensionality_comparison']
            
            methods = []
            structure_scores = []
            
            if 'structure_preservation' in dim_results:
                for method, scores in dim_results['structure_preservation'].items():
                    if 'spearman_correlation' in scores:
                        methods.append(method)
                        structure_scores.append(abs(scores['spearman_correlation']))
            
            if methods:
                bars = ax.bar(methods, structure_scores, color='lightsteelblue', alpha=0.7)
                ax.set_title('降维方法结构保持能力', fontweight='bold')
                ax.set_ylabel('Spearman相关系数')
                ax.set_xticklabels(methods, rotation=45)
                
                # 添加数值标签
                for bar, score in zip(bars, structure_scores):
                    height = bar.get_height()
                    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                        f'{score:.3f}', ha='center', va='bottom')
        
        # 13. 内在维度vs原始维度 (3,0)
        ax = axes[3, 0]
        if 'high_dimensional_analysis' in enhanced_results:
            high_dim = enhanced_results['high_dimensional_analysis']
            if 'intrinsic_dimensionality' in high_dim:
                intrinsic_dim = high_dim['intrinsic_dimensionality']
                original_dim = 341
                
                categories = ['内在维度', '原始维度']
                dimensions = [intrinsic_dim, original_dim]
                colors = ['lightcoral', 'lightblue']
                
                bars = ax.bar(categories, dimensions, color=colors, alpha=0.7)
                ax.set_title('维度对比分析', fontweight='bold')
                ax.set_ylabel('维度数')
                ax.set_yscale('log')
                
                # 添加数值标签
                for bar, dim in zip(bars, dimensions):
                    height = bar.get_height()
                    ax.text(bar.get_x() + bar.get_width()/2., height * 1.1,
                        f'{dim:.1f}', ha='center', va='bottom')
        
        # 14. 高维分类器性能对比 (3,1)
        ax = axes[3, 1]
        if 'high_dimensional_analysis' in enhanced_results:
            high_dim = enhanced_results['high_dimensional_analysis']
            if 'separability_scores' in high_dim:
                sep_scores = high_dim['separability_scores']
                
                classifiers = list(sep_scores.keys())
                scores = [sep_scores[clf]['mean_score'] for clf in classifiers]
                errors = [sep_scores[clf]['std_score'] for clf in classifiers]
                
                bars = ax.bar(classifiers, scores, yerr=errors, capsize=5, 
                            color='lightgreen', alpha=0.7, error_kw={'linewidth': 2})
                ax.set_title('高维分类器性能对比', fontweight='bold')
                ax.set_ylabel('准确率')
                ax.set_xticklabels(classifiers, rotation=45)
                
                # 添加随机基线
                n_subjects = len(self.data['available_subjects'])
                baseline = 1.0 / n_subjects
                ax.axhline(baseline, color='red', linestyle='--', label=f'随机基线: {baseline:.3f}')
                ax.legend()
        
        # 15. 特征组重要性对比 (3,2)
        ax = axes[3, 2]
        if 'feature_group_analysis' in enhanced_results:
            group_results = enhanced_results['feature_group_analysis']
            
            groups = []
            variances = []
            
            for group_name, group_info in group_results.items():
                if 'variance_in_3pc' in group_info:
                    groups.append(group_name)
                    variances.append(group_info['variance_in_3pc'])
            
            if groups:
                bars = ax.bar(groups, variances, color='wheat', alpha=0.7)
                ax.set_title('特征组前3PC解释方差', fontweight='bold')
                ax.set_ylabel('解释方差比例')
                ax.set_xticklabels(groups, rotation=45)
                
                # 添加数值标签
                for bar, var in zip(bars, variances):
                    height = bar.get_height()
                    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                        f'{var:.3f}', ha='center', va='bottom')
        
        # 16. 距离度量对比 (3,3)
        ax = axes[3, 3]
        if 'high_dimensional_analysis' in enhanced_results:
            high_dim = enhanced_results['high_dimensional_analysis']
            if 'distance_matrices' in high_dim:
                distance_matrices = high_dim['distance_matrices']
                
                metrics = list(distance_matrices.keys())
                mean_distances = []
                
                for metric in metrics:
                    dist_matrix = distance_matrices[metric]
                    upper_triangle = dist_matrix[np.triu_indices_from(dist_matrix, k=1)]
                    mean_distances.append(np.mean(upper_triangle))
                
                bars = ax.bar(metrics, mean_distances, color='plum', alpha=0.7)
                ax.set_title('不同距离度量对比', fontweight='bold')
                ax.set_ylabel('平均距离')
                ax.set_xticklabels(metrics, rotation=45)
        
        # 17. 聚类结果可视化 (3,4)
        ax = axes[3, 4]
        if 'high_dimensional_analysis' in enhanced_results:
            high_dim = enhanced_results['high_dimensional_analysis']
            if 'clustering_results' in high_dim:
                clustering_results = high_dim['clustering_results']
                
                cluster_methods = list(clustering_results.keys())
                n_clusters = [clustering_results[method]['n_clusters'] for method in cluster_methods]
                n_noise = [clustering_results[method]['n_noise'] for method in cluster_methods]
                
                x = np.arange(len(cluster_methods))
                width = 0.35
                
                bars1 = ax.bar(x - width/2, n_clusters, width, label='聚类数', color='skyblue', alpha=0.7)
                bars2 = ax.bar(x + width/2, n_noise, width, label='噪点数', color='salmon', alpha=0.7)
                
                ax.set_title('DBSCAN聚类结果', fontweight='bold')
                ax.set_ylabel('数量')
                ax.set_xticks(x)
                ax.set_xticklabels([method.replace('dbscan_eps_', 'ε=') for method in cluster_methods])
                ax.legend()

    def _generate_dimensionality_comparison_plot(self):
        """生成专门的降维方法对比图"""
        if 'enhanced_dimensionality' not in self.analysis_results:
            return
        
        enhanced_results = self.analysis_results['enhanced_dimensionality']
        
        if 'dimensionality_comparison' not in enhanced_results:
            return
        
        dim_results = enhanced_results['dimensionality_comparison']
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        
        # PCA结果
        if 'pca' in dim_results and dim_results['pca'] is not None:
            ax = axes[0, 0]
            pca_2d = dim_results['pca']['2d_result']
            scatter = ax.scatter(pca_2d[:, 0], pca_2d[:, 1], 
                            c=range(len(pca_2d)), cmap='viridis', s=100, alpha=0.7)
            ax.set_title('PCA降维结果', fontweight='bold', fontsize=14)
            ax.set_xlabel('PC1')
            ax.set_ylabel('PC2')
            plt.colorbar(scatter, ax=ax)
        
        # Kernel PCA结果  
        if 'kernel_pca' in dim_results and dim_results['kernel_pca'] is not None:
            ax = axes[0, 1]
            kpca_2d = dim_results['kernel_pca']['rbf_2d_result']
            scatter = ax.scatter(kpca_2d[:, 0], kpca_2d[:, 1],
                            c=range(len(kpca_2d)), cmap='viridis', s=100, alpha=0.7)
            ax.set_title('Kernel PCA (RBF)降维结果', fontweight='bold', fontsize=14)
            ax.set_xlabel('KPC1')
            ax.set_ylabel('KPC2')
            plt.colorbar(scatter, ax=ax)
        
        # t-SNE结果
        if 'tsne' in dim_results and dim_results['tsne'] is not None:
            ax = axes[0, 2]
            tsne_2d = dim_results['tsne']['2d_result']
            scatter = ax.scatter(tsne_2d[:, 0], tsne_2d[:, 1],
                            c=range(len(tsne_2d)), cmap='viridis', s=100, alpha=0.7)
            ax.set_title('t-SNE降维结果', fontweight='bold', fontsize=14)
            ax.set_xlabel('t-SNE 1')
            ax.set_ylabel('t-SNE 2')
            plt.colorbar(scatter, ax=ax)
        
        # UMAP结果
        if 'umap' in dim_results and dim_results['umap'] is not None:
            ax = axes[1, 0]
            umap_2d = dim_results['umap']['2d_result']
            scatter = ax.scatter(umap_2d[:, 0], umap_2d[:, 1],
                            c=range(len(umap_2d)), cmap='viridis', s=100, alpha=0.7)
            ax.set_title('UMAP降维结果', fontweight='bold', fontsize=14)
            ax.set_xlabel('UMAP 1')
            ax.set_ylabel('UMAP 2')
            plt.colorbar(scatter, ax=ax)
        
        # PCA解释方差
        if 'pca' in dim_results:
            ax = axes[1, 1]
            explained_var = dim_results['pca']['explained_variance'][:20]  # 前20个
            ax.plot(range(1, len(explained_var)+1), explained_var, 'o-', linewidth=2, markersize=6)
            ax.set_title('PCA解释方差', fontweight='bold', fontsize=14)
            ax.set_xlabel('主成分')
            ax.set_ylabel('解释方差比例')
            ax.grid(True, alpha=0.3)
        
        # 结构保持能力对比
        if 'structure_preservation' in dim_results:
            ax = axes[1, 2]
            methods = []
            correlations = []
            
            for method, scores in dim_results['structure_preservation'].items():
                if 'spearman_correlation' in scores:
                    methods.append(method)
                    correlations.append(abs(scores['spearman_correlation']))
            
            if methods:
                bars = ax.bar(methods, correlations, color='lightcoral', alpha=0.7)
                ax.set_title('结构保持能力对比', fontweight='bold', fontsize=14)
                ax.set_ylabel('Spearman相关系数')
                ax.set_xticklabels(methods, rotation=45)
                
                # 添加数值标签
                for bar, corr in zip(bars, correlations):
                    height = bar.get_height()
                    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                        f'{corr:.3f}', ha='center', va='bottom')
        
        plt.tight_layout()
        dim_viz_path = os.path.join(self.save_path, 'visualizations', 'dimensionality_comparison.png')
        plt.savefig(dim_viz_path, dpi=300, bbox_inches='tight')
        plt.show()
        
        print(f"  ✅ 降维对比图已保存: {dim_viz_path}")

    def _generate_feature_group_plot(self):
        """生成特征分组分析图"""
        if 'enhanced_dimensionality' not in self.analysis_results:
            return
        
        enhanced_results = self.analysis_results['enhanced_dimensionality']
        
        if 'feature_group_analysis' not in enhanced_results:
            return
        
        group_results = enhanced_results['feature_group_analysis']
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        
        # 1. 各组解释方差对比
        ax = axes[0, 0]
        groups = []
        variances_3pc = []
        feature_counts = []
        
        for group_name, group_info in group_results.items():
            if 'variance_in_3pc' in group_info:
                groups.append(group_name)
                variances_3pc.append(group_info['variance_in_3pc'])
                feature_counts.append(group_info['feature_count'])
        
        if groups:
            bars = ax.bar(groups, variances_3pc, color='wheat', alpha=0.7)
            ax.set_title('各特征组前3PC解释方差', fontweight='bold', fontsize=14)
            ax.set_ylabel('解释方差比例')
            ax.set_xticklabels(groups, rotation=45)
            
            # 添加特征数量标签
            for bar, var, count in zip(bars, variances_3pc, feature_counts):
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                    f'{var:.3f}\n({count}特征)', ha='center', va='bottom')
        
        # 2. 各组受试者间距离对比
        ax = axes[0, 1]
        if groups:
            mean_distances = []
            cv_distances = []
            
            for group_name in groups:
                if group_name in group_results and 'mean_distance' in group_results[group_name]:
                    mean_distances.append(group_results[group_name]['mean_distance'])
                    cv_distances.append(group_results[group_name]['coefficient_of_variation'])
            
            if mean_distances:
                bars = ax.bar(groups, mean_distances, color='lightblue', alpha=0.7)
                ax.set_title('各特征组平均受试者间距离', fontweight='bold', fontsize=14)
                ax.set_ylabel('平均距离')
                ax.set_xticklabels(groups, rotation=45)
        
        # 3. 各组分离性能力对比
        ax = axes[1, 0]
        if groups:
            separabilities = []
            
            for group_name in groups:
                if group_name in group_results and 'separability_score' in group_results[group_name]:
                    separabilities.append(group_results[group_name]['separability_score'])
                else:
                    separabilities.append(0)
            
            bars = ax.bar(groups, separabilities, color='lightgreen', alpha=0.7)
            ax.set_title('各特征组受试者分离能力', fontweight='bold', fontsize=14)
            ax.set_ylabel('分离性能得分')
            ax.set_xticklabels(groups, rotation=45)
            
            # 添加数值标签
            for bar, sep in zip(bars, separabilities):
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                    f'{sep:.3f}', ha='center', va='bottom')
        
        # 4. 特征重要性分布（如果有高维分析结果）
        ax = axes[1, 1]
        if 'high_dimensional_analysis' in enhanced_results:
            high_dim = enhanced_results['high_dimensional_analysis']
            if 'feature_importance' in high_dim:
                feature_importance = high_dim['feature_importance']
                
                # 按特征组显示重要性分布
                group_ranges = {
                    'qti_params': (0, 15),
                    'raw_b_tensors': (15, 225),
                    'cest_params': (225, 229),
                    'z_spectrum': (229, 341)
                }
                
                group_importances = {}
                for group_name, (start, end) in group_ranges.items():
                    if end <= len(feature_importance):
                        group_imp = feature_importance[start:end]
                        group_importances[group_name] = np.mean(group_imp)
                
                if group_importances:
                    groups_imp = list(group_importances.keys())
                    importances = list(group_importances.values())
                    
                    bars = ax.bar(groups_imp, importances, color='salmon', alpha=0.7)
                    ax.set_title('各特征组平均重要性', fontweight='bold', fontsize=14)
                    ax.set_ylabel('平均重要性得分')
                    ax.set_xticklabels(groups_imp, rotation=45)
        
        plt.tight_layout()
        group_viz_path = os.path.join(self.save_path, 'visualizations', 'feature_group_analysis.png')
        plt.savefig(group_viz_path, dpi=300, bbox_inches='tight')
        plt.show()
        
        print(f"  ✅ 特征分组图已保存: {group_viz_path}")


    def phase4_decision_generation(self):
        """
        Phase 4: 基于所有分析结果生成最终决策和建议 - 增强版
        """
        print("\n" + "="*80)
        print("🎯 Phase 4: 决策建议生成 (增强版)")
        print("="*80)
        
        # 综合所有决策得分
        all_scores = {}
        for phase in self.decision_scores:
            all_scores.update(self.decision_scores[phase])
        
        # 增强版决策权重（加入新指标）
        weights = {
            'difference_significance': 0.12,
            'pattern_linearity': 0.15,
            'subject_separability': 0.15,
            'class_consistency': 0.12,
            'sample_adequacy': 0.10,
            'embedding_feasibility': 0.15,
            'intrinsic_dimensionality': 0.15,  # 新增
            'high_dim_performance': 0.06       # 新增
        }
        
        # 计算加权总分
        weighted_score = sum(all_scores.get(key, 0) * weight 
                        for key, weight in weights.items())
        
        # 生成增强版具体建议
        recommendations = []
        embedding_suggestions = {}
        
        # 基于增强分析结果生成建议
        enhanced_results = self.analysis_results.get('enhanced_dimensionality', {})
        
        # 1. 差异显著性建议
        if all_scores.get('difference_significance', 0) > 0.7:
            recommendations.append("✅ 受试者间差异显著，Subject Embedding很有必要")
        elif all_scores.get('difference_significance', 0) > 0.3:
            recommendations.append("📊 受试者间存在一定差异，Subject Embedding有帮助")
        else:
            recommendations.append("⚠️ 受试者间差异较小，Subject Embedding效果可能有限")
        
        # 2. 基于内在维度的建议
        intrinsic_dim_score = all_scores.get('intrinsic_dimensionality', 0.5)
        if 'high_dimensional_analysis' in enhanced_results:
            intrinsic_dim = enhanced_results['high_dimensional_analysis'].get('intrinsic_dimensionality', 341)
            
            if intrinsic_dim < 20:
                recommendations.append(f"✅ 内在维度很低({intrinsic_dim:.1f})，embedding效果预期很好")
                embedding_suggestions['embedding_dim'] = max(16, min(32, int(intrinsic_dim * 1.5)))
            elif intrinsic_dim < 50:
                recommendations.append(f"📊 内在维度中等({intrinsic_dim:.1f})，embedding有明显帮助")
                embedding_suggestions['embedding_dim'] = max(32, min(64, int(intrinsic_dim * 2)))
            elif intrinsic_dim < 100:
                recommendations.append(f"⚠️ 内在维度较高({intrinsic_dim:.1f})，embedding需要较高维度")
                embedding_suggestions['embedding_dim'] = max(64, min(128, int(intrinsic_dim * 2.5)))
            else:
                recommendations.append(f"❌ 内在维度过高({intrinsic_dim:.1f})，可能需要特征选择")
                embedding_suggestions['embedding_dim'] = min(256, max(128, int(intrinsic_dim * 2)))
                embedding_suggestions['feature_selection_needed'] = True
        
        # 3. 基于降维分析的建议
        if 'dimensionality_comparison' in enhanced_results:
            dim_results = enhanced_results['dimensionality_comparison']
            
            if 'pca' in dim_results:
                pca_3pc = np.sum(dim_results['pca']['explained_variance'][:3])
                
                if pca_3pc > 0.8:
                    recommendations.append("✅ PCA前3PC解释方差>80%，线性embedding足够")
                    embedding_suggestions['embedding_type'] = 'linear'
                elif pca_3pc > 0.6:
                    recommendations.append("📊 PCA前3PC解释方差>60%，线性embedding为主")
                    embedding_suggestions['embedding_type'] = 'mixed_linear'
                else:
                    recommendations.append("⚠️ PCA前3PC解释方差<60%，考虑非线性embedding")
                    embedding_suggestions['embedding_type'] = 'nonlinear'
            
            # 结构保持分析
            if 'structure_preservation' in dim_results:
                preservation_scores = []
                for method, scores in dim_results['structure_preservation'].items():
                    if 'spearman_correlation' in scores:
                        preservation_scores.append(abs(scores['spearman_correlation']))
                
                if preservation_scores:
                    avg_preservation = np.mean(preservation_scores)
                    if avg_preservation > 0.8:
                        recommendations.append("✅ 结构保持能力强，降维方法有效")
                    elif avg_preservation > 0.6:
                        recommendations.append("📊 结构保持能力中等，需要选择合适的embedding方法")
                    else:
                        recommendations.append("⚠️ 结构保持能力较弱，建议高维embedding")
        
        # 4. 基于特征分组的建议
        if 'feature_group_analysis' in enhanced_results:
            group_results = enhanced_results['feature_group_analysis']
            group_variances = {}
            
            for group_name, group_info in group_results.items():
                if 'variance_in_3pc' in group_info:
                    group_variances[group_name] = group_info['variance_in_3pc']
            
            if group_variances:
                best_group = max(group_variances, key=group_variances.get)
                best_variance = group_variances[best_group]
                
                recommendations.append(f"🔍 最佳特征组: {best_group} (解释方差: {best_variance:.3f})")
                
                if best_variance > 0.8:
                    recommendations.append("✅ 存在高质量特征组，可考虑分组embedding")
                    embedding_suggestions['use_feature_groups'] = True
                    embedding_suggestions['best_feature_group'] = best_group
        
        # 5. 基于高维分析的建议
        if 'high_dimensional_analysis' in enhanced_results:
            high_dim = enhanced_results['high_dimensional_analysis']
            
            if 'separability_scores' in high_dim:
                sep_scores = high_dim['separability_scores']
                if sep_scores:
                    avg_separability = np.mean([scores['mean_score'] for scores in sep_scores.values()])
                    n_subjects = len(self.data['available_subjects'])
                    baseline = 1.0 / n_subjects
                    
                    if avg_separability > baseline * 3:
                        recommendations.append("⚠️ 受试者高度可分离，需要特征去偏差设计")
                        embedding_suggestions['use_debiasing'] = True
                    elif avg_separability > baseline * 2:
                        recommendations.append("📊 受试者中度可分离，建议平衡embedding设计")
            
            # 聚类分析建议
            if 'clustering_results' in high_dim:
                clustering_results = high_dim['clustering_results']
                if clustering_results:
                    best_clustering = None
                    best_quality = 0
                    
                    for method, result in clustering_results.items():
                        n_clusters = result['n_clusters']
                        n_noise = result['n_noise']
                        total_points = len(result['labels'])
                        
                        if n_clusters > 1 and n_noise / total_points < 0.3:
                            quality = n_clusters / total_points
                            if quality > best_quality:
                                best_quality = quality
                                best_clustering = (method, n_clusters)
                    
                    if best_clustering:
                        method, n_clusters = best_clustering
                        recommendations.append(f"🔍 发现{n_clusters}个受试者聚类，可考虑Mixture of Experts")
                        embedding_suggestions['use_mixture_experts'] = True
                        embedding_suggestions['n_expert_clusters'] = n_clusters
        
        # 6. 样本量和计算资源建议
        sample_adequacy = all_scores.get('sample_adequacy', 0)
        if sample_adequacy > 0.7:
            recommendations.append("✅ 样本量充足，支持复杂embedding方法")
            embedding_suggestions['can_use_complex_methods'] = True
        elif sample_adequacy > 0.4:
            recommendations.append("📊 样本量中等，建议中等复杂度embedding")
            embedding_suggestions['complexity_level'] = 'medium'
        else:
            recommendations.append("⚠️ 样本量偏少，建议简单embedding方法")
            embedding_suggestions['complexity_level'] = 'simple'
        
        # 7. 最终决策生成
        if weighted_score > 0.75:
            final_decision = "强烈推荐使用Subject Embedding"
            confidence = "高"
        elif weighted_score > 0.6:
            final_decision = "建议使用Subject Embedding" 
            confidence = "中高"
        elif weighted_score > 0.45:
            final_decision = "可以尝试Subject Embedding"
            confidence = "中"
        elif weighted_score > 0.3:
            final_decision = "Subject Embedding可能有帮助"
            confidence = "低"
        else:
            final_decision = "不建议使用Subject Embedding"
            confidence = "高"
        
        # 8. 生成具体实施建议
        implementation_suggestions = self._generate_implementation_suggestions(
            embedding_suggestions, all_scores, enhanced_results
        )
        
        # 9. 新受试者适应策略建议
        adaptation_strategy = self._suggest_adaptation_strategy(
            all_scores, enhanced_results, embedding_suggestions
        )
        
        # 保存增强版决策结果
        decision_result = {
            'weighted_score': weighted_score,
            'final_decision': final_decision,
            'confidence': confidence,
            'recommendations': recommendations,
            'embedding_suggestions': embedding_suggestions,
            'implementation_suggestions': implementation_suggestions,
            'adaptation_strategy': adaptation_strategy,
            'detailed_scores': all_scores,
            'analysis_summary': self._generate_analysis_summary(enhanced_results)
        }
        
        self.analysis_results['enhanced_final_decision'] = decision_result
        
        # 打印增强版决策报告
        self._print_enhanced_decision_report(decision_result)
        
        return decision_result

    def _generate_implementation_suggestions(self, embedding_suggestions, all_scores, enhanced_results):
        """生成具体的实施建议"""
        suggestions = []
        
        # 基础实施建议
        embedding_dim = embedding_suggestions.get('embedding_dim', 64)
        suggestions.append(f"推荐embedding维度: {embedding_dim}")
        
        # Embedding类型建议
        embedding_type = embedding_suggestions.get('embedding_type', 'linear')
        if embedding_type == 'linear':
            suggestions.extend([
                "使用简单的nn.Embedding层",
                "学习率建议: 1e-3到1e-4",
                "添加L2正则化防止过拟合"
            ])
        elif embedding_type == 'mixed_linear':
            suggestions.extend([
                "使用nn.Embedding + 小型MLP",
                "MLP建议: embedding_dim -> embedding_dim*2 -> embedding_dim",
                "学习率建议: 1e-4到1e-5"
            ])
        else:  # nonlinear
            suggestions.extend([
                "使用深度MLP进行非线性embedding",
                "MLP建议: embedding_dim -> embedding_dim*4 -> embedding_dim*2 -> embedding_dim",
                "使用批归一化和Dropout"
            ])
        
        # 特征处理建议
        if embedding_suggestions.get('feature_selection_needed', False):
            suggestions.append("强烈建议进行特征选择，保留最重要的50-70%特征")
        
        if embedding_suggestions.get('use_feature_groups', False):
            best_group = embedding_suggestions.get('best_feature_group', '')
            suggestions.append(f"可优先使用{best_group}特征组进行embedding")
        
        # 去偏差建议
        if embedding_suggestions.get('use_debiasing', False):
            suggestions.extend([
                "实施特征去偏差策略",
                "考虑对抗训练: 最大化分类性能，最小化受试者识别",
                "使用梯度反转层(Gradient Reversal Layer)"
            ])
        
        # 混合专家建议
        if embedding_suggestions.get('use_mixture_experts', False):
            n_experts = embedding_suggestions.get('n_expert_clusters', 3)
            suggestions.extend([
                f"考虑Mixture of Experts架构，使用{n_experts}个专家",
                "先对受试者进行聚类，再训练每个专家",
                "使用门控网络动态选择专家"
            ])
        
        # 训练策略建议
        complexity = embedding_suggestions.get('complexity_level', 'medium')
        if complexity == 'simple':
            suggestions.extend([
                "使用较小的batch size (64-128)",
                "早停patience建议: 5-10 epochs",
                "使用简单的学习率调度"
            ])
        elif complexity == 'medium':
            suggestions.extend([
                "使用中等batch size (128-256)",
                "早停patience建议: 10-15 epochs", 
                "可尝试余弦退火学习率调度"
            ])
        else:  # complex
            suggestions.extend([
                "使用较大batch size (256-512)",
                "早停patience建议: 15-20 epochs",
                "使用warmup + 余弦退火策略"
            ])
        
        return suggestions

    def _suggest_adaptation_strategy(self, all_scores, enhanced_results, embedding_suggestions):
        """建议新受试者适应策略"""
        strategy = {
            'primary_method': '',
            'backup_methods': [],
            'required_samples': 0,
            'expected_performance': '',
            'implementation_notes': []
        }
        
        # 基于受试者可分离性选择主要策略
        separability = all_scores.get('subject_separability', 0.5)
        
        if separability > 0.7:
            # 高可分离性：需要快速适应
            strategy['primary_method'] = '少样本快速适应'
            strategy['required_samples'] = 500
            strategy['backup_methods'] = ['相似性迁移', '零样本泛化']
            strategy['expected_performance'] = '85-95%的最优性能'
            strategy['implementation_notes'] = [
                "冻结主模型参数，只训练新受试者embedding",
                "使用较高学习率(1e-3)快速收敛",
                "监控过拟合，通常10-50轮即可"
            ]
        
        elif separability > 0.4:
            # 中等可分离性：相似性迁移为主
            strategy['primary_method'] = '相似性迁移 + 微调'
            strategy['required_samples'] = 200
            strategy['backup_methods'] = ['少样本适应', '平均embedding初始化']
            strategy['expected_performance'] = '75-85%的最优性能'
            strategy['implementation_notes'] = [
                "先找最相似的受试者继承embedding",
                "用少量样本进行微调",
                "相似性可基于特征统计量判断"
            ]
        
        else:
            # 低可分离性：零样本泛化
            strategy['primary_method'] = '零样本泛化'
            strategy['required_samples'] = 0
            strategy['backup_methods'] = ['平均embedding', '回归预测embedding']
            strategy['expected_performance'] = '60-75%的最优性能'
            strategy['implementation_notes'] = [
                "使用所有训练受试者embedding的均值",
                "或基于特征统计量回归预测embedding",
                "无需新受试者的标注数据"
            ]
        
        # 基于聚类结果调整策略
        if embedding_suggestions.get('use_mixture_experts', False):
            strategy['implementation_notes'].append(
                "如果使用Mixture of Experts，需要先确定新受试者属于哪个专家组"
            )
        
        # 基于内在维度调整样本需求
        if 'high_dimensional_analysis' in enhanced_results:
            intrinsic_dim = enhanced_results['high_dimensional_analysis'].get('intrinsic_dimensionality', 100)
            if intrinsic_dim < 20:
                strategy['required_samples'] = max(50, strategy['required_samples'] // 2)
                strategy['implementation_notes'].append("内在维度低，需要样本量可以减半")
            elif intrinsic_dim > 100:
                strategy['required_samples'] = strategy['required_samples'] * 2
                strategy['implementation_notes'].append("内在维度高，建议增加样本量")
        
        return strategy

    def _generate_analysis_summary(self, enhanced_results):
        """生成分析总结"""
        summary = {
            'data_characteristics': {},
            'key_findings': [],
            'technical_insights': []
        }
        
        # 数据特征总结
        if 'high_dimensional_analysis' in enhanced_results:
            high_dim = enhanced_results['high_dimensional_analysis']
            summary['data_characteristics'] = {
                'intrinsic_dimensionality': high_dim.get('intrinsic_dimensionality', 'unknown'),
                'distance_concentration': high_dim.get('distance_concentration', 'unknown'),
                'clustering_feasibility': len(high_dim.get('clustering_results', {})) > 0
            }
        
        # 关键发现
        if 'dimensionality_comparison' in enhanced_results:
            dim_results = enhanced_results['dimensionality_comparison']
            if 'pca' in dim_results:
                pca_10pc = np.sum(dim_results['pca']['explained_variance'][:10])
                summary['key_findings'].append(f"前10个主成分解释{pca_10pc*100:.1f}%的方差")
                
            if 'structure_preservation' in dim_results:
                preservation_scores = []
                for method, scores in dim_results['structure_preservation'].items():
                    if 'spearman_correlation' in scores:
                        preservation_scores.append(abs(scores['spearman_correlation']))
                
                if preservation_scores:
                    avg_preservation = np.mean(preservation_scores)
                    summary['key_findings'].append(f"降维方法平均保持{avg_preservation*100:.1f}%的结构信息")
        
        # 技术洞察
        if 'feature_group_analysis' in enhanced_results:
            group_results = enhanced_results['feature_group_analysis']
            group_qualities = {}
            
            for group_name, group_info in group_results.items():
                if 'variance_in_3pc' in group_info:
                    group_qualities[group_name] = group_info['variance_in_3pc']
            
            if group_qualities:
                best_group = max(group_qualities, key=group_qualities.get)
                worst_group = min(group_qualities, key=group_qualities.get)
                summary['technical_insights'].append(
                    f"特征组质量差异显著: {best_group}最佳({group_qualities[best_group]:.3f}), "
                    f"{worst_group}最差({group_qualities[worst_group]:.3f})"
                )
        
        return summary

    def _print_enhanced_decision_report(self, decision_result):
        """打印增强版决策报告"""
        print(f"\n📋 Subject Embedding 增强版可行性分析报告")
        print("=" * 70)
        print(f"🎯 最终决策: {decision_result['final_decision']}")
        print(f"🎲 置信度: {decision_result['confidence']}")
        print(f"📊 综合得分: {decision_result['weighted_score']:.3f} / 1.0")
        
        print(f"\n💡 主要发现:")
        for i, rec in enumerate(decision_result['recommendations'], 1):
            print(f"  {i:2d}. {rec}")
        
        print(f"\n🔧 Embedding设计建议:")
        embedding_suggestions = decision_result['embedding_suggestions']
        for key, value in embedding_suggestions.items():
            if isinstance(value, bool):
                if value:
                    print(f"  ✅ {key.replace('_', ' ').title()}")
            else:
                print(f"  📋 {key.replace('_', ' ').title()}: {value}")
        
        print(f"\n🛠️ 实施建议:")
        for i, suggestion in enumerate(decision_result['implementation_suggestions'], 1):
            print(f"  {i:2d}. {suggestion}")
        
        print(f"\n🎯 新受试者适应策略:")
        adaptation = decision_result['adaptation_strategy']
        print(f"  🥇 主要方法: {adaptation['primary_method']}")
        print(f"  📊 所需样本: {adaptation['required_samples']} 个体素")
        print(f"  🎯 预期性能: {adaptation['expected_performance']}")
        print(f"  🔄 备选方法: {', '.join(adaptation['backup_methods'])}")
        
        if adaptation['implementation_notes']:
            print(f"  💡 实施要点:")
            for note in adaptation['implementation_notes']:
                print(f"     • {note}")
        
        print(f"\n📈 详细得分:")
        for key, score in decision_result['detailed_scores'].items():
            print(f"  - {key.replace('_', ' ').title()}: {score:.3f}")
        
        # 分析总结
        summary = decision_result.get('analysis_summary', {})
        if summary:
            print(f"\n🔍 数据特征总结:")
            data_chars = summary.get('data_characteristics', {})
            for key, value in data_chars.items():
                print(f"  - {key.replace('_', ' ').title()}: {value}")
            
            if 'key_findings' in summary and summary['key_findings']:
                print(f"\n🎯 关键发现:")
                for finding in summary['key_findings']:
                    print(f"  • {finding}")
            
            if 'technical_insights' in summary and summary['technical_insights']:
                print(f"\n💡 技术洞察:")
                for insight in summary['technical_insights']:
                    print(f"  • {insight}")
                    
    def generate_report(self):
        """生成完整的增强版分析报告"""
        report_path = os.path.join(self.save_path, 'enhanced_subject_embedding_analysis_report.txt')
        
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write("Subject Embedding 增强版可行性分析报告\n")
            f.write("=" * 70 + "\n\n")
            f.write(f"生成时间: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"分析版本: 增强版 v2.0 (包含高维分析、降维对比、特征分组)\n\n")
            
            # 数据概况
            f.write("📊 数据概况\n")
            f.write("-" * 30 + "\n")
            f.write(f"受试者数量: {len(self.data['available_subjects'])}\n")
            f.write(f"特征维度: {self.data['X_train'].shape[1]}\n")
            f.write(f"训练样本量: {len(self.data['X_train']):,}\n")
            f.write(f"验证样本量: {len(self.data['X_val']):,}\n")
            f.write(f"测试样本量: {len(self.data['X_test']):,}\n\n")
            
            # 增强版分析结果
            if 'enhanced_dimensionality' in self.analysis_results:
                enhanced_results = self.analysis_results['enhanced_dimensionality']
                
                f.write("🔍 增强版维度分析结果\n")
                f.write("-" * 30 + "\n")
                
                # 内在维度分析
                if 'high_dimensional_analysis' in enhanced_results:
                    high_dim = enhanced_results['high_dimensional_analysis']
                    intrinsic_dim = high_dim.get('intrinsic_dimensionality', 'unknown')
                    f.write(f"估计内在维度: {intrinsic_dim}\n")
                    
                    if 'distance_concentration' in high_dim:
                        concentration = high_dim['distance_concentration']
                        f.write(f"距离集中度: {concentration:.3f}\n")
                    
                    if 'separability_scores' in high_dim and high_dim['separability_scores']:
                        avg_sep = np.mean([scores['mean_score'] for scores in high_dim['separability_scores'].values()])
                        f.write(f"高维分类器平均性能: {avg_sep:.3f}\n")
                
                # 降维方法对比
                if 'dimensionality_comparison' in enhanced_results:
                    dim_results = enhanced_results['dimensionality_comparison']
                    
                    f.write(f"\n降维方法对比:\n")
                    if 'pca' in dim_results:
                        pca_3pc = np.sum(dim_results['pca']['explained_variance'][:3])
                        pca_10pc = np.sum(dim_results['pca']['explained_variance'][:10])
                        f.write(f"  PCA前3PC解释方差: {pca_3pc:.3f}\n")
                        f.write(f"  PCA前10PC解释方差: {pca_10pc:.3f}\n")
                    
                    if 'structure_preservation' in dim_results:
                        f.write(f"  结构保持能力评估:\n")
                        for method, scores in dim_results['structure_preservation'].items():
                            if 'spearman_correlation' in scores:
                                corr = scores['spearman_correlation']
                                f.write(f"    {method}: {abs(corr):.3f}\n")
                
                # 特征分组分析
                if 'feature_group_analysis' in enhanced_results:
                    group_results = enhanced_results['feature_group_analysis']
                    f.write(f"\n特征分组分析:\n")
                    
                    for group_name, group_info in group_results.items():
                        if 'variance_in_3pc' in group_info:
                            f.write(f"  {group_name}:\n")
                            f.write(f"    特征数量: {group_info['feature_count']}\n")
                            f.write(f"    前3PC解释方差: {group_info['variance_in_3pc']:.3f}\n")
                            if 'separability_score' in group_info:
                                f.write(f"    分离性能: {group_info['separability_score']:.3f}\n")
            
            # 最终决策（使用增强版结果）
            f.write(f"\n🎯 增强版最终决策\n")
            f.write("-" * 30 + "\n")
            
            if 'enhanced_final_decision' in self.analysis_results:
                decision = self.analysis_results['enhanced_final_decision']
                f.write(f"决策: {decision['final_decision']}\n")
                f.write(f"置信度: {decision['confidence']}\n")
                f.write(f"综合得分: {decision['weighted_score']:.3f}\n\n")
                
                # Embedding设计建议
                f.write("Embedding设计建议:\n")
                embedding_suggestions = decision['embedding_suggestions']
                for key, value in embedding_suggestions.items():
                    f.write(f"  {key.replace('_', ' ').title()}: {value}\n")
                
                # 实施建议
                f.write(f"\n实施建议:\n")
                for suggestion in decision['implementation_suggestions']:
                    f.write(f"  • {suggestion}\n")
                
                # 新受试者适应策略
                f.write(f"\n新受试者适应策略:\n")
                adaptation = decision['adaptation_strategy']
                f.write(f"  主要方法: {adaptation['primary_method']}\n")
                f.write(f"  所需样本: {adaptation['required_samples']}\n")
                f.write(f"  预期性能: {adaptation['expected_performance']}\n")
                f.write(f"  备选方法: {', '.join(adaptation['backup_methods'])}\n")
                
                # 分析总结
                if 'analysis_summary' in decision:
                    summary = decision['analysis_summary']
                    f.write(f"\n分析总结:\n")
                    
                    if 'key_findings' in summary:
                        f.write("  关键发现:\n")
                        for finding in summary['key_findings']:
                            f.write(f"    • {finding}\n")
                    
                    if 'technical_insights' in summary:
                        f.write("  技术洞察:\n")
                        for insight in summary['technical_insights']:
                            f.write(f"    • {insight}\n")
            
            # 建议后续工作
            f.write(f"\n📋 建议后续工作\n")
            f.write("-" * 30 + "\n")
            f.write("1. 基于分析结果实施Subject Embedding\n")
            f.write("2. 进行对比实验验证embedding效果\n")
            f.write("3. 测试不同新受试者适应策略\n")
            f.write("4. 优化embedding维度和架构\n")
            f.write("5. 在更大数据集上验证结论稳定性\n")
        
            print(f"✅ 增强版完整报告已保存: {report_path}")
            return report_path

# ============================================================================
# 🚀 主执行函数
# ============================================================================
def run_subject_embedding_analysis_enhanced(data_path='/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38.mat',
                                          train_subjects_range=(1, 31),
                                          val_subjects_range=(31, 38), 
                                          test_subject=38,
                                          save_path='./subject_embedding_analysis/',
                                          random_state=42):
    """
    运行增强版Subject Embedding可行性分析 - Multi-Subject-Out策略
    
    Args:
        data_path: 数据文件路径
        train_subjects_range: 训练集受试者范围 (start, end)
        val_subjects_range: 验证集受试者范围 (start, end)
        test_subject: 测试集受试者ID  
        save_path: 结果保存路径
        random_state: 随机种子
        
    Returns:
        analyzer: 分析器对象
        decision_result: 决策结果
        data_dict: 完整数据字典
    """
    
    print("🧠 开始Subject Embedding增强版可行性分析...")
    print("="*80)
    print(f"📊 分析策略: Multi-Subject-Out")
    print(f"🎯 训练集: 受试者{train_subjects_range[0]}-{train_subjects_range[1]-1}")
    print(f"🎯 验证集: 受试者{val_subjects_range[0]}-{val_subjects_range[1]-1}")
    print(f"🎯 测试集: 受试者{test_subject}")
    
    # Phase 0: 增强版数据准备
    data_dict = load_and_prepare_data_multi_subject_out(
        data_path=data_path,
        train_subjects_range=train_subjects_range,
        val_subjects_range=val_subjects_range,
        test_subject=test_subject,
        random_state=random_state
    )
    
    # 初始化分析器
    analyzer = SubjectEmbeddingAnalyzer(save_path=save_path)
    
    # 使用增强版数据准备
    data = analyzer.prepare_data_with_subjects_enhanced(data_dict)
    
    if data is None:
        print("❌ 数据准备失败，分析终止")
        return None, None, None
    
    # Phase 1-4: 保持原有分析流程
    analyzer.phase1_subject_differences_analysis()
    analyzer.phase2_subject_separability_analysis()
    analyzer.phase3_embedding_adaptability_analysis()
    
    # 生成可视化和决策
    analyzer.generate_visualizations()
    decision_result = analyzer.phase4_decision_generation()
    
    # 生成报告
    analyzer.generate_report()
    
    print("\n" + "="*80)
    print("🎉 Subject Embedding增强版分析完成!")
    print("="*80)
    print(f"📁 所有结果保存在: {save_path}")
    print(f"🎯 最终建议: {decision_result['final_decision']}")
    print(f"📊 综合得分: {decision_result['weighted_score']:.3f}")
    
    return analyzer, decision_result, data_dict

# ============================================================================
# 💡 使用示例
# ============================================================================

# 推荐的增强版使用方式
analyzer, decision, data_dict = run_subject_embedding_analysis_enhanced(
    data_path='/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38.mat',
    train_subjects_range=(1, 31),    # 受试者1-30训练
    val_subjects_range=(31, 38),     # 受试者31-37验证
    test_subject=38,                 # 受试者38测试
    save_path='./subject_embedding_analysis_enhanced/'
)

# 查看增强版分析结果
if decision:
    print("\n🎯 增强版分析结果:")
    print("最终建议:", decision['final_decision'])
    print("建议embedding维度:", decision.get('embedding_suggestions', {}).get('embedding_dim', 'N/A'))
    print("适应策略:", decision.get('adaptation_strategy', {}).get('primary_method', 'N/A'))

# 访问完整数据用于后续实验
X_train_scaled = data_dict['X_train_scaled']
y_train = data_dict['y_train']
subjects_train = data_dict['subjects_train']

X_val_scaled = data_dict['X_val_scaled'] 
y_val = data_dict['y_val']
subjects_val = data_dict['subjects_val']

X_test_scaled = data_dict['X_test_scaled']
y_test = data_dict['y_test']
subjects_test = data_dict['subjects_test']

print(f"\n📊 数据完整性验证:")
print(f"训练集受试者: {sorted(np.unique(subjects_train))}")
print(f"验证集受试者: {sorted(np.unique(subjects_val))}")
print(f"测试集受试者: {sorted(np.unique(subjects_test))}")

# 查看决策结果
print("最终建议:", decision['final_decision'])
print("建议embedding维度:", decision['embedding_dim_suggestion'])
print("实施建议:", decision['implementation_suggestions'])

# 访问详细分析结果
print("受试者识别准确率:", analyzer.analysis_results['subject_identification']['accuracy'])
print("PCA前3PC解释方差:", np.sum(analyzer.analysis_results['feature_variation']['pca_explained_variance'][:3]))


🧠 开始Subject Embedding可行性分析...
🧠 Subject Embedding 可行性分析器初始化完成
📁 结果保存路径: ./subject_embedding_analysis/

📊 Phase 0: 精确数据准备与受试者ID重建
🔄 加载原始数据进行精确映射...
